# Notebook 05 — Truth Matching, Splits, and Supervision Table

**Responsibilities (plan §12.6):**
- Candidate-to-truth matching via one-to-one Hungarian assignment (§9.1)
- `candidate_to_truth_match.parquet` (§3.7)
- `splits.parquet` — spatial block cross-validation (§3.10, §10.2)
- `training_supervision_table.parquet` (§3.9)

**This notebook does NOT:**
- Run any detectors
- Compute features
- Train models
- Generate annotations

## Matching policy (§9.1)
- **Algorithm:** Hungarian (optimal one-to-one) with hard distance threshold `truth_match_radius_px`
- **Hard negatives:** `EXPLICIT_NEGATIVE` annotations yield `matched_hard_negative`
- **Negatives** derived only from crops allowed for negative supervision
- **False negatives** (unmatched truth) tracked for evaluation, never trained on
- **Truth coordinates:** `refined_click_well_y/x_px` preferred; falls back to `raw_click_well_y/x_px` with `truth_coord_source` recorded

## Splits (§10.2)
- Spatial block CV at crop level (each crop = one block)
- Stratified by well_id: 60% train / 20% calibration / 20% test
- Deterministic hash-based assignment — reproducible across runs
- Regenerated deterministically by default to avoid stale split files

## Supervision table (§3.9)
- Left-join of `mega_feature_table` + match results + splits
- **One row per union_id** — every union candidate gets a row
- `supervision_status` ∈ {`included`, `excluded`}; `target_binary` ∈ {0, 1, NaN}
- `sample_weight` computed via configurable strategy
- Candidates outside annotated territory → `supervision_status = excluded`, `target_binary = NaN`

## Robustness improvements in this version
- Prefers a single coherent NB03/04 run manifest when loading disk outputs
- Avoids stale-table mixing across different upstream runs
- Idempotent metadata merges (safe on rerun)
- Off-channel negatives are force-labeled correctly without duplicate rows
- Deterministic split regeneration avoids stale cached split leakage


In [10]:
# -------------------------------------------------------------------
# REPO DISCOVERY
# -------------------------------------------------------------------
from pathlib import Path

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / ".git").exists() or (p / "registries").exists():
            return p
    raise RuntimeError("Could not locate repo root from current working directory.")

REPO_ROOT = find_repo_root()
print("REPO_ROOT =", REPO_ROOT)

REPO_ROOT = /Users/baghnatios/Desktop/spot-detection-meta


In [11]:
# -------------------------------------------------------------------
# CONFIG
# -------------------------------------------------------------------
CFG = {
    # ── Execution / provenance ─────────────────────────────────────────────
    "execution_mode": "full_run",
    "repo_root": str(REPO_ROOT),
    "upstream_nb34_run_id": None,

    # ── Explicit path overrides (set to a concrete path string to bypass autodiscovery) ──
    "candidate_union_path_override": None,
    "mega_feature_table_path_override": None,
    "candidate_raw_path_override": None,
    "annotations_path_override": None,
    "crop_registry_path_override": None,
    "candidate_clusters_path_override": None,
    "candidate_union_membership_path_override": None,
    "off_channel_negative_candidates_path_override": None,
    "detector_cache_dir_override": None,
    "feature_family_cache_dir_override": None,
    "offtrain_detector_cache_dir_override": None,
    "offtrain_feature_cache_dir_override": None,
    "offaware_aug_cache_dir_override": None,

    # ── Upstream cache handling / validation ───────────────────────────────
    "load_detector_method_caches": True,
    "load_feature_family_caches": True,
    "load_offtrain_detector_caches": True,
    "load_offtrain_feature_caches": True,
    "load_offaware_aug_caches": True,
    "join_offaware_aug_into_training_supervision": True,
    "strict_upstream_cache_validation": False,
    "write_upstream_cache_audit": True,


    # ── Directory layout ───────────────────────────────────────────────────
    "tables_dir": "tables",
    "crop_registry_dir": "annotations/crop_registry",
    "clicked_truth_dir": "annotations/clicked_truth",
    "artifact_report_dir": "artifacts/reports/nb05_truth_matching",
    "manifest_dir": "artifacts/manifests",

    # ── Matching policy ────────────────────────────────────────────────────
    "matching_algorithm": "hungarian_thresholded",
    "matching_registry_version": "matching_v1_crop_hungarian_medoid_refined_halo",
    "truth_coordinate_policy": "refined_well_fallback_raw",
    "candidate_coordinate_policy": "union_medoid_well",
    "truth_match_radius_px": 8.0,

    # ── EXPLICIT_NEGATIVE matching ─────────────────────────────────────────
    # Candidates within this radius of an EXPLICIT_NEGATIVE annotation
    # get match_status="matched_hard_negative" (target=0, higher weight)
    "explicit_negative_match_radius_px": 8.0,
    "hard_negative_weight_multiplier": 3.0,

    # ── Negative generation ────────────────────────────────────────────────
    # Any crop that has at least one annotation (TRUE_SPOT or EXPLICIT_NEGATIVE)
    # is treated as "completed" for negative generation purposes.
    # This fixes the bug where annotator_status="pending" blocked all negatives.
    "negative_generation_policy": "any_annotated_crop",
    "min_annotations_for_negative_generation": 1,

    # ── Splits (§10.2) ────────────────────────────────────────────────────
    "split_train_frac": 0.60,
    "split_cal_frac": 0.20,
    "split_test_frac": 0.20,
    "split_seed": 42,
    "split_registry_version": "v1_nb05",
    # If True and a splits.parquet with matching split_registry_version exists, reuse it
    "reuse_existing_splits": False,

    # ── Sample weighting ──────────────────────────────────────────────────
    # "balanced"          : w = n_total / (2 * n_class)
    # "sqrt_inverse_freq" : w = 1/sqrt(class_freq), normalized to mean=1
    # "none"              : w = 1.0 for all supervised rows
    "sample_weight_strategy": "balanced",
    "positive_sample_weight_override": None,   # set to float to skip strategy
    "negative_sample_weight_override": None,
    "excluded_sample_weight": 0.0,

    # ── annotation_coverage_status values (from mega) treated as supervised territory ──
    "accepted_annotation_coverage_statuses_for_negatives": [
        "fully_annotated_crop", "annotated_crop", "complete", "in_crop"
    ],
    "use_mega_annotation_coverage_if_available": True,

    # ── Persistence flags ──────────────────────────────────────────────────
    "write_candidate_to_truth_match": True,
    "write_training_supervision_table": True,
    "write_truth_evaluation_summary": True,
    "write_splits": True,
    "write_reports": True,

    # ── QA / reporting ─────────────────────────────────────────────────────
    "qa_max_overlay_crops": 8,
    "qa_point_alpha": 0.9,
    "qa_fig_dpi": 160,
    "write_match_diagnostics": True,
    "min_annotations_per_crop_warn": 1,
}

CFG

{'execution_mode': 'full_run',
 'repo_root': '/Users/baghnatios/Desktop/spot-detection-meta',
 'upstream_nb34_run_id': None,
 'candidate_union_path_override': None,
 'mega_feature_table_path_override': None,
 'candidate_raw_path_override': None,
 'annotations_path_override': None,
 'crop_registry_path_override': None,
 'candidate_clusters_path_override': None,
 'candidate_union_membership_path_override': None,
 'off_channel_negative_candidates_path_override': None,
 'detector_cache_dir_override': None,
 'feature_family_cache_dir_override': None,
 'offtrain_detector_cache_dir_override': None,
 'offtrain_feature_cache_dir_override': None,
 'offaware_aug_cache_dir_override': None,
 'load_detector_method_caches': True,
 'load_feature_family_caches': True,
 'load_offtrain_detector_caches': True,
 'load_offtrain_feature_caches': True,
 'load_offaware_aug_caches': True,
 'join_offaware_aug_into_training_supervision': True,
 'strict_upstream_cache_validation': False,
 'write_upstream_cache_aud

In [12]:
# -------------------------------------------------------------------
# IMPORTS
# -------------------------------------------------------------------
import gc
import hashlib
import json
import math
import os
import re
import shutil
import time
from collections import defaultdict, Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import tifffile
import yaml

from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree

print("imports ok")

imports ok


In [13]:
# -------------------------------------------------------------------
# LOGGING / IO / PROVENANCE HELPERS
# -------------------------------------------------------------------
def ts_utc() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

def log(msg: str) -> None:
    print(f"[{ts_utc()}] {msg}", flush=True)

def ensure_dir(path) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p

def make_run_id(prefix: str = "nb05") -> str:
    t = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    h = hashlib.sha1(f"{t}|{os.getpid()}|nb05".encode("utf-8")).hexdigest()[:8]
    return f"{prefix}_{t}_{h}"

def sha1_text(x: str) -> str:
    return hashlib.sha1(x.encode("utf-8")).hexdigest()

def safe_to_parquet(df: pd.DataFrame, path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(path, index=False)
    except Exception as e:
        csv_path = path.with_suffix(".csv")
        log(f"[warn] parquet write failed ({e}); writing CSV fallback -> {csv_path.name}")
        df.to_csv(csv_path, index=False)
        return csv_path
    return path

def write_json(obj, path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, sort_keys=False), encoding="utf-8")
    return path

def read_yaml_or_json(path):
    path = Path(path)
    txt = path.read_text(encoding="utf-8")
    if path.suffix.lower() in {".yaml", ".yml"}:
        return yaml.safe_load(txt)
    return json.loads(txt)

def latest_matching_file(search_dir: Path, patterns: list):
    """Return the most-recently-modified file matching any glob pattern in search_dir."""
    candidates = []
    if not search_dir.exists():
        return None
    for pat in patterns:
        candidates.extend(search_dir.glob(pat))
    candidates = [p for p in candidates if p.is_file()]
    if not candidates:
        return None
    candidates.sort(key=lambda p: (p.stat().st_mtime, str(p)), reverse=True)
    return candidates[0]

def latest_nb34_manifest(manifest_dir: Path):
    manifests = []
    if manifest_dir.exists():
        manifests.extend(manifest_dir.glob("*run_manifest.json"))
    manifests = [p for p in manifests if p.is_file()]
    scored = []
    for p in manifests:
        try:
            payload = json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            continue
        nb_name = str(payload.get("notebook_name", "")).lower()
        paths = payload.get("paths", {})
        if "candidate_union" in paths and ("candidate_features" in paths or "mega_feature_table" in paths):
            score = 1
            if "nb03" in nb_name or "nb04" in nb_name or "candidate_universe" in nb_name:
                score = 2
            scored.append((score, p.stat().st_mtime, str(p), payload, p))
    if not scored:
        return None, None
    scored.sort(reverse=True)
    _, _, _, payload, path = scored[0]
    return path, payload

RUN_ID = make_run_id("nb05")
TABLES_DIR   = ensure_dir(REPO_ROOT / CFG["tables_dir"])
REPORT_DIR   = ensure_dir(REPO_ROOT / CFG["artifact_report_dir"])
MANIFEST_DIR = ensure_dir(REPO_ROOT / CFG["manifest_dir"])

log(f"TABLES_DIR   = {TABLES_DIR}")
log(f"REPORT_DIR   = {REPORT_DIR}")
log(f"MANIFEST_DIR = {MANIFEST_DIR}")
log(f"RUN_ID = {RUN_ID}")


[2026-01-24 04:26:57 UTC] TABLES_DIR   = /Users/baghnatios/Desktop/spot-detection-meta/tables
[2026-01-24 04:26:57 UTC] REPORT_DIR   = /Users/baghnatios/Desktop/spot-detection-meta/artifacts/reports/nb05_truth_matching
[2026-01-24 04:26:57 UTC] MANIFEST_DIR = /Users/baghnatios/Desktop/spot-detection-meta/artifacts/manifests
[2026-01-24 04:26:57 UTC] RUN_ID = nb05_20260124T042657Z_2af2592e


In [14]:
# -------------------------------------------------------------------
# CLEAN NB05 OUTPUTS FROM PRIOR RUNS
# Deletes only Notebook 05-derived artifacts so reruns cannot mix stale tables,
# manifests, or report images from earlier matching/split regimes.
# -------------------------------------------------------------------
log("=" * 90)
log("NB05 STALE-OUTPUT CLEANUP")

NB05_TABLE_PATTERNS = [
    "nb05_*_candidate_to_truth_match.parquet",
    "nb05_*_training_supervision_table.parquet",
    "nb05_*_truth_evaluation_summary.parquet",
    "nb05_*_splits.parquet",
    "nb05_*_matching_timing.parquet",
    "nb05_*_crop_match_summary.parquet",
]
NB05_MANIFEST_PATTERNS = [
    "nb05_*_truth_matching_manifest.json",
]
NB05_REPORT_SUBDIRS = [
    REPORT_DIR / "qa_overlays",
]

deleted_paths = []

for pat in NB05_TABLE_PATTERNS:
    for p in sorted(TABLES_DIR.glob(pat)):
        if p.is_file():
            p.unlink()
            deleted_paths.append(p)

for pat in NB05_MANIFEST_PATTERNS:
    for p in sorted(MANIFEST_DIR.glob(pat)):
        if p.is_file():
            p.unlink()
            deleted_paths.append(p)

for p in NB05_REPORT_SUBDIRS:
    if p.exists():
        shutil.rmtree(p, ignore_errors=True)
        deleted_paths.append(p)

log(f"Deleted {len(deleted_paths)} stale NB05 artifact paths.")
for p in deleted_paths[:20]:
    log(f"  removed: {p}")
if len(deleted_paths) > 20:
    log(f"  ... plus {len(deleted_paths) - 20} more")


[2026-01-24 04:26:57 UTC] ==========================================================================================
[2026-01-24 04:26:57 UTC] NB05 STALE-OUTPUT CLEANUP
[2026-01-24 04:26:57 UTC] Deleted 0 stale NB05 artifact paths.


In [15]:
# -------------------------------------------------------------------
# DISCOVERY / IN-MEMORY RESOLUTION
#
# Priority for each upstream NB03/04 artifact:
#   1. Explicit CFG path override
#   2. In-memory object already in kernel
#   3. Preferred coherent NB03/04 run manifest (if upstream_nb34_run_id set)
#   4. Latest coherent NB03/04 run manifest
#   5. Fallback latest matching file in tables/
#
# Annotations are special:
#   - Prefer per-crop NB02 annotation parquet files under annotations/clicked_truth/
#   - Fall back to tables/annotations.parquet only if no per-crop files exist
# -------------------------------------------------------------------
tables_dir        = REPO_ROOT / CFG["tables_dir"]
crop_registry_dir = REPO_ROOT / CFG["crop_registry_dir"]
clicked_truth_dir = REPO_ROOT / CFG["clicked_truth_dir"]

def select_nb34_manifest(manifest_dir: Path, preferred_run_id: Optional[str] = None):
    manifests = []
    if manifest_dir.exists():
        manifests.extend(manifest_dir.glob("*run_manifest.json"))
    manifests = [p for p in manifests if p.is_file()]
    scored = []
    for p in manifests:
        try:
            payload = json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            continue
        nb_name = str(payload.get("notebook_name", "")).lower()
        run_id  = str(payload.get("run_id", "")).strip()
        paths   = payload.get("paths", {})
        if "candidate_union" not in paths:
            continue
        if ("candidate_features" not in paths) and ("mega_feature_table" not in paths):
            continue
        score = 1
        if "nb03" in nb_name or "nb04" in nb_name or "candidate_universe" in nb_name:
            score = 2
        prefer = int(bool(preferred_run_id) and run_id == str(preferred_run_id).strip())
        created_at = str(payload.get("created_at_utc", "") or payload.get("run_id", ""))
        scored.append((prefer, score, created_at, str(p), payload, p))
    if not scored:
        return None, None
    scored.sort(reverse=True)
    _, _, _, _, payload, path = scored[0]
    return path, payload

def latest_matching_file(base: Path, patterns):
    if base is None or not Path(base).exists():
        return None
    hits = []
    for pat in patterns:
        hits.extend(list(Path(base).glob(pat)))
    hits = [p for p in hits if p.is_file()]
    if not hits:
        return None
    hits.sort(key=lambda p: (p.stat().st_mtime, str(p)))
    return hits[-1]

def prefer_run_glob(patterns, preferred_run_id=None):
    hits = []
    if tables_dir.exists():
        for pat in patterns:
            hits.extend(list(tables_dir.glob(pat)))
    hits = [p for p in hits if p.is_file()]
    if not hits:
        return None
    if preferred_run_id:
        tagged = [p for p in hits if str(preferred_run_id) in p.name]
        if tagged:
            tagged.sort(key=lambda p: (p.stat().st_mtime, str(p)))
            return tagged[-1]
    hits.sort(key=lambda p: (p.stat().st_mtime, str(p)))
    return hits[-1]

preferred_nb34_run_id = str(CFG.get("upstream_nb34_run_id") or "").strip() or None
nb34_manifest_path, nb34_manifest = select_nb34_manifest(MANIFEST_DIR, preferred_nb34_run_id)
nb34_paths = (nb34_manifest or {}).get("paths", {}) if nb34_manifest else {}

if nb34_manifest_path is not None:
    _sel_run_id = str((nb34_manifest or {}).get("run_id", ""))
    log(f"Using upstream manifest candidate: {nb34_manifest_path.name}")
    if preferred_nb34_run_id:
        log(f"Preferred upstream NB03/04 run_id: {preferred_nb34_run_id}")
        assert _sel_run_id == preferred_nb34_run_id, (
            f"Resolved NB03/04 manifest run_id={_sel_run_id}, expected {preferred_nb34_run_id}. "
            "Refusing to continue with a mismatched upstream run."
        )
else:
    log("No coherent NB03/04 manifest found; falling back to tables/ file discovery.")

def manifest_path_for(key):
    p = nb34_paths.get(key, None) if nb34_paths else None
    return Path(p) if p else None

_input_sources = {}

# ── candidate_union ───────────────────────────────────────────────────────
if CFG["candidate_union_path_override"]:
    candidate_union_path = Path(CFG["candidate_union_path_override"])
    _input_sources["candidate_union"] = f"override:{candidate_union_path}"
else:
    try:
        _ = candidate_union
        candidate_union_path = None
        _input_sources["candidate_union"] = "in_memory"
    except NameError:
        candidate_union_path = manifest_path_for("candidate_union")
        if candidate_union_path is not None:
            _input_sources["candidate_union"] = f"manifest:{candidate_union_path}"
        else:
            candidate_union_path = prefer_run_glob(["*candidate_union.parquet"], preferred_nb34_run_id)
            _input_sources["candidate_union"] = f"disk:{candidate_union_path}"

# ── mega_feature_table / candidate_features ───────────────────────────────
if CFG["mega_feature_table_path_override"]:
    mega_feature_table_path = Path(CFG["mega_feature_table_path_override"])
    _input_sources["mega_feature_table"] = f"override:{mega_feature_table_path}"
else:
    try:
        _ = mega_feature_table
        mega_feature_table_path = None
        _input_sources["mega_feature_table"] = "in_memory"
    except NameError:
        try:
            _ = features_df
            mega_feature_table_path = None
            _input_sources["mega_feature_table"] = "in_memory:features_df"
        except NameError:
            mega_feature_table_path = manifest_path_for("candidate_features")
            if mega_feature_table_path is None:
                mega_feature_table_path = manifest_path_for("mega_feature_table")
            if mega_feature_table_path is not None:
                _input_sources["mega_feature_table"] = f"manifest:{mega_feature_table_path}"
            else:
                mega_feature_table_path = prefer_run_glob(["*candidate_features.parquet", "*mega_feature_table*.parquet"], preferred_nb34_run_id)
                _input_sources["mega_feature_table"] = f"disk:{mega_feature_table_path}"

# ── candidate_raw ─────────────────────────────────────────────────────────
if CFG["candidate_raw_path_override"]:
    candidate_raw_path = Path(CFG["candidate_raw_path_override"])
    _input_sources["candidate_raw"] = f"override:{candidate_raw_path}"
else:
    try:
        _ = candidate_raw
        candidate_raw_path = None
        _input_sources["candidate_raw"] = "in_memory"
    except NameError:
        candidate_raw_path = manifest_path_for("candidate_raw")
        if candidate_raw_path is not None:
            _input_sources["candidate_raw"] = f"manifest:{candidate_raw_path}"
        else:
            candidate_raw_path = prefer_run_glob(["*candidate_raw.parquet"], preferred_nb34_run_id)
            _input_sources["candidate_raw"] = f"disk:{candidate_raw_path}"

# ── candidate_clusters ────────────────────────────────────────────────────
if CFG["candidate_clusters_path_override"]:
    candidate_clusters_path = Path(CFG["candidate_clusters_path_override"])
    _input_sources["candidate_clusters"] = f"override:{candidate_clusters_path}"
else:
    try:
        _ = candidate_clusters
        candidate_clusters_path = None
        _input_sources["candidate_clusters"] = "in_memory"
    except NameError:
        candidate_clusters_path = manifest_path_for("candidate_clusters")
        if candidate_clusters_path is not None:
            _input_sources["candidate_clusters"] = f"manifest:{candidate_clusters_path}"
        else:
            candidate_clusters_path = prefer_run_glob(["*candidate_clusters.parquet"], preferred_nb34_run_id)
            _input_sources["candidate_clusters"] = f"disk:{candidate_clusters_path}"

# ── candidate_union_membership ────────────────────────────────────────────
if CFG["candidate_union_membership_path_override"]:
    candidate_union_membership_path = Path(CFG["candidate_union_membership_path_override"])
    _input_sources["candidate_union_membership"] = f"override:{candidate_union_membership_path}"
else:
    try:
        _ = candidate_union_membership
        candidate_union_membership_path = None
        _input_sources["candidate_union_membership"] = "in_memory"
    except NameError:
        candidate_union_membership_path = manifest_path_for("candidate_union_membership")
        if candidate_union_membership_path is not None:
            _input_sources["candidate_union_membership"] = f"manifest:{candidate_union_membership_path}"
        else:
            candidate_union_membership_path = prefer_run_glob(["*candidate_union_membership.parquet"], preferred_nb34_run_id)
            _input_sources["candidate_union_membership"] = f"disk:{candidate_union_membership_path}"

# ── off_channel_negative_candidates ───────────────────────────────────────
if CFG["off_channel_negative_candidates_path_override"]:
    off_channel_negative_candidates_path = Path(CFG["off_channel_negative_candidates_path_override"])
    _input_sources["off_channel_negative_candidates"] = f"override:{off_channel_negative_candidates_path}"
else:
    try:
        _ = off_channel_negative_candidates
        off_channel_negative_candidates_path = None
        _input_sources["off_channel_negative_candidates"] = "in_memory"
    except NameError:
        off_channel_negative_candidates_path = manifest_path_for("off_channel_negative_candidates")
        if off_channel_negative_candidates_path is not None:
            _input_sources["off_channel_negative_candidates"] = f"manifest:{off_channel_negative_candidates_path}"
        else:
            off_channel_negative_candidates_path = prefer_run_glob(["*off_channel_negative_candidates.parquet"], preferred_nb34_run_id)
            _input_sources["off_channel_negative_candidates"] = f"disk:{off_channel_negative_candidates_path}"

# ── annotations (NB02 canonical) ──────────────────────────────────────────
# Prefer all per-crop annotation parquet files if present.
_annotation_glob_hits = []
if clicked_truth_dir.exists():
    _annotation_glob_hits.extend(sorted(clicked_truth_dir.rglob("*_annotations.parquet")))
    if not _annotation_glob_hits:
        _annotation_glob_hits.extend(sorted(clicked_truth_dir.rglob("*.parquet")))

if CFG["annotations_path_override"]:
    annotations_path = Path(CFG["annotations_path_override"])
    _input_sources["annotations"] = f"override:{annotations_path}"
elif _annotation_glob_hits:
    annotations_path = [Path(p) for p in _annotation_glob_hits if Path(p).is_file()]
    _input_sources["annotations"] = f"clicked_truth_dir:{len(annotations_path)} files"
else:
    annotations_path = tables_dir / "annotations.parquet" if (tables_dir / "annotations.parquet").exists() else None
    _input_sources["annotations"] = f"disk:{annotations_path}"

# ── crop_registry ─────────────────────────────────────────────────────────
if CFG["crop_registry_path_override"]:
    crop_registry_path = Path(CFG["crop_registry_path_override"])
    _input_sources["crop_registry"] = f"override:{crop_registry_path}"
else:
    try:
        _ = crop_registry
        crop_registry_path = None
        _input_sources["crop_registry"] = "in_memory"
    except NameError:
        crop_registry_path = manifest_path_for("crop_registry")
        if crop_registry_path is None:
            search_candidates = []
            for base in [crop_registry_dir, tables_dir, REPO_ROOT / "artifacts", REPO_ROOT]:
                if base.exists():
                    search_candidates.extend(list(base.glob("*crop_registry*.parquet")))
                    search_candidates.extend(list(base.glob("*crop_registry*.yaml")))
                    search_candidates.extend(list(base.glob("*crop_registry*.yml")))
            search_candidates = [p for p in search_candidates if p.is_file()]
            if search_candidates:
                if preferred_nb34_run_id:
                    tagged = [p for p in search_candidates if preferred_nb34_run_id in p.name]
                    if tagged:
                        search_candidates = tagged
                search_candidates.sort(key=lambda p: (p.stat().st_mtime, str(p)))
                crop_registry_path = search_candidates[-1]
        _input_sources["crop_registry"] = f"{'manifest' if manifest_path_for('crop_registry') is not None else 'disk'}:{crop_registry_path}"

# cache dirs
detector_cache_dir = Path(CFG["detector_cache_dir_override"]) if CFG["detector_cache_dir_override"] else manifest_path_for("detector_cache_dir")
feature_family_cache_dir = Path(CFG["feature_family_cache_dir_override"]) if CFG["feature_family_cache_dir_override"] else manifest_path_for("feature_family_cache_dir")
offtrain_detector_cache_dir = Path(CFG["offtrain_detector_cache_dir_override"]) if CFG["offtrain_detector_cache_dir_override"] else None
offtrain_feature_cache_dir  = Path(CFG["offtrain_feature_cache_dir_override"]) if CFG["offtrain_feature_cache_dir_override"] else None
offaware_aug_cache_dir      = Path(CFG["offaware_aug_cache_dir_override"]) if CFG["offaware_aug_cache_dir_override"] else None

resolved_paths = {
    "candidate_raw":             str(candidate_raw_path) if candidate_raw_path else "(in_memory)",
    "candidate_union":           str(candidate_union_path) if candidate_union_path else "(in_memory)",
    "mega_feature_table":        str(mega_feature_table_path) if mega_feature_table_path else "(in_memory)",
    "candidate_clusters":        str(candidate_clusters_path) if candidate_clusters_path else "(in_memory)",
    "candidate_union_membership": str(candidate_union_membership_path) if candidate_union_membership_path else "(in_memory)",
    "off_channel_negative_candidates": str(off_channel_negative_candidates_path) if off_channel_negative_candidates_path else "(in_memory)",
    "annotations":               [str(p) for p in annotations_path] if isinstance(annotations_path, list) else (str(annotations_path) if annotations_path else None),
    "crop_registry":             str(crop_registry_path) if crop_registry_path else "(in_memory)",
    "nb34_manifest":             str(nb34_manifest_path) if nb34_manifest_path else None,
    "preferred_nb34_run_id":     preferred_nb34_run_id,
}
log("Input resolution:")
for k, src in _input_sources.items():
    log(f"  {k}: {src}")

assert annotations_path is not None and ((isinstance(annotations_path, list) and len(annotations_path) > 0) or not isinstance(annotations_path, list)), (
    "No annotations file found. Expected per-crop NB02 parquet files in annotations/clicked_truth/ "
    "or a fallback tables/annotations.parquet. Run NB02 first."
)
for _name, _path in [
    ("candidate_union", candidate_union_path),
    ("mega_feature_table", mega_feature_table_path),
    ("crop_registry", crop_registry_path),
]:
    if _path is not None:
        assert Path(_path).exists(), (
            f"Required input not found on disk: {_name} -> {_path}. "
            "Check path overrides or run upstream notebooks."
        )

if preferred_nb34_run_id:
    assert off_channel_negative_candidates_path is not None and Path(off_channel_negative_candidates_path).exists(), (
        "Expected off_channel_negative_candidates side table for the bound upstream NB03/04 run, "
        f"but could not resolve one for run_id={preferred_nb34_run_id}."
    )


[2026-01-24 04:26:57 UTC] Using upstream manifest candidate: nb03_20260124T024537Z_066ded0c_run_manifest.json
[2026-01-24 04:26:57 UTC] Input resolution:
[2026-01-24 04:26:57 UTC]   candidate_union: in_memory
[2026-01-24 04:26:57 UTC]   mega_feature_table: in_memory
[2026-01-24 04:26:57 UTC]   candidate_raw: in_memory
[2026-01-24 04:26:57 UTC]   candidate_clusters: in_memory
[2026-01-24 04:26:57 UTC]   candidate_union_membership: in_memory
[2026-01-24 04:26:57 UTC]   off_channel_negative_candidates: in_memory
[2026-01-24 04:26:57 UTC]   annotations: clicked_truth_dir:50 files
[2026-01-24 04:26:57 UTC]   crop_registry: in_memory


In [16]:
print("candidate_union in globals:", "candidate_union" in dir())
try:
    print("in-memory candidate_union shape:", candidate_union.shape)
except NameError:
    print("not in memory")
print()
print("candidate_union_path:", candidate_union_path)

candidate_union in globals: True
in-memory candidate_union shape: (5955, 34)

candidate_union_path: None


In [17]:
# -------------------------------------------------------------------
# LOAD INPUT TABLES
# Only reads from disk for tables not already resolved in-memory above.
# Tables already in kernel scope are used directly (no disk I/O).
# -------------------------------------------------------------------
log("Resolving input tables ...")
t0_load = time.perf_counter()

def _read_parquet_if_exists(path_obj):
    p = Path(path_obj)
    return pd.read_parquet(p) if p.exists() else pd.DataFrame()

def _load_annotations_source(src):
    if src is None:
        return pd.DataFrame()
    if isinstance(src, (list, tuple)):
        frames = []
        for p in src:
            p = Path(p)
            if p.exists():
                try:
                    df = pd.read_parquet(p)
                    if len(df):
                        frames.append(df)
                except Exception as e:
                    log(f"[warn] failed to read annotation parquet {p.name}: {type(e).__name__}: {e}")
        if not frames:
            return pd.DataFrame()
        out = pd.concat(frames, ignore_index=True, sort=False)
        return out
    p = Path(src)
    return pd.read_parquet(p) if p.exists() else pd.DataFrame()

# candidate_raw
if candidate_raw_path is not None:
    candidate_raw = _read_parquet_if_exists(candidate_raw_path)
    log(f"  candidate_raw               : loaded from disk ({len(candidate_raw):,} rows)")
else:
    try:
        candidate_raw = candidate_raw.copy()
    except NameError:
        candidate_raw = pd.DataFrame()
    log(f"  candidate_raw               : from kernel ({len(candidate_raw):,} rows)")

# candidate_union
if candidate_union_path is not None:
    candidate_union = _read_parquet_if_exists(candidate_union_path)
    log(f"  candidate_union             : loaded from disk ({len(candidate_union):,} rows)")
else:
    try:
        candidate_union = candidate_union.copy()
    except NameError:
        candidate_union = pd.DataFrame()
    log(f"  candidate_union             : from kernel ({len(candidate_union):,} rows)")

# mega_feature_table
if mega_feature_table_path is not None:
    mega_feature_table = _read_parquet_if_exists(mega_feature_table_path)
    log(f"  mega_feature_table          : loaded from disk ({len(mega_feature_table):,} rows)")
else:
    try:
        mega_feature_table = mega_feature_table.copy()
    except NameError:
        try:
            mega_feature_table = features_df.copy()
        except NameError:
            try:
                mega_feature_table = mega.copy()
            except NameError:
                mega_feature_table = pd.DataFrame()
    log(f"  mega_feature_table          : from kernel ({len(mega_feature_table):,} rows)")

# annotations — always from disk
annotations = _load_annotations_source(annotations_path)
if isinstance(annotations_path, (list, tuple)):
    log(f"  annotations                 : loaded from disk ({len(annotations):,} rows from {len(annotations_path)} files)")
else:
    log(f"  annotations                 : loaded from disk ({len(annotations):,} rows)")

# candidate_clusters
if candidate_clusters_path is not None:
    candidate_clusters = _read_parquet_if_exists(candidate_clusters_path)
    log(f"  candidate_clusters          : loaded from disk ({len(candidate_clusters):,} rows)")
else:
    try:
        candidate_clusters = candidate_clusters.copy()
    except NameError:
        candidate_clusters = pd.DataFrame()
    log(f"  candidate_clusters          : from kernel ({len(candidate_clusters):,} rows)")

# candidate_union_membership
if candidate_union_membership_path is not None:
    candidate_union_membership = _read_parquet_if_exists(candidate_union_membership_path)
    log(f"  candidate_union_membership  : loaded from disk ({len(candidate_union_membership):,} rows)")
else:
    try:
        candidate_union_membership = candidate_union_membership.copy()
    except NameError:
        candidate_union_membership = pd.DataFrame()
    log(f"  candidate_union_membership  : from kernel ({len(candidate_union_membership):,} rows)")

# off_channel_negative_candidates (side table)
if off_channel_negative_candidates_path is not None:
    off_channel_negative_candidates = _read_parquet_if_exists(off_channel_negative_candidates_path)
    log(f"  off_channel_negative_candidates : loaded from disk ({len(off_channel_negative_candidates):,} rows)")
else:
    try:
        off_channel_negative_candidates = off_channel_negative_candidates.copy()
    except NameError:
        off_channel_negative_candidates = pd.DataFrame()
    log(f"  off_channel_negative_candidates : from kernel ({len(off_channel_negative_candidates):,} rows)")

# crop_registry
if crop_registry_path is not None:
    _cr_ext = Path(crop_registry_path).suffix.lower()
    if _cr_ext == ".parquet":
        crop_registry = pd.read_parquet(crop_registry_path)
    elif _cr_ext in {".yaml", ".yml"}:
        _payload = read_yaml_or_json(crop_registry_path)
        crop_registry = pd.DataFrame(
            _payload["records"] if isinstance(_payload, dict) and "records" in _payload else _payload
        )
    else:
        raise ValueError(f"Unsupported crop registry extension: {_cr_ext}")
    log(f"  crop_registry               : loaded from disk ({len(crop_registry):,} rows)")
else:
    try:
        crop_registry = crop_registry.copy()
    except NameError:
        crop_registry = pd.DataFrame()
    log(f"  crop_registry               : from kernel ({len(crop_registry):,} rows)")

log(f"  resolved in {time.perf_counter() - t0_load:.2f}s")


[2026-01-24 04:26:57 UTC] Resolving input tables ...
[2026-01-24 04:26:57 UTC]   candidate_raw               : from kernel (55,814 rows)
[2026-01-24 04:26:57 UTC]   candidate_union             : from kernel (5,955 rows)
[2026-01-24 04:26:57 UTC]   mega_feature_table          : from kernel (5,955 rows)
[2026-01-24 04:26:58 UTC]   annotations                 : loaded from disk (1,795 rows from 50 files)
[2026-01-24 04:26:58 UTC]   candidate_clusters          : from kernel (5,365 rows)
[2026-01-24 04:26:58 UTC]   candidate_union_membership  : from kernel (55,814 rows)
[2026-01-24 04:26:58 UTC]   off_channel_negative_candidates : from kernel (16,269 rows)
[2026-01-24 04:26:58 UTC]   crop_registry               : from kernel (50 rows)
[2026-01-24 04:26:58 UTC]   resolved in 0.12s


In [18]:

# -------------------------------------------------------------------
# LOAD / VALIDATE UPSTREAM NB03/04 CACHE ARTIFACTS
#
# NB03/04 now writes reusable disk caches for:
#   - per-method detector outputs
#   - per-family feature tables
#   - panel preprocessing / response-map caches
#
# NB05 still trains only on the canonical full-well tables:
#   candidate_union, mega_feature_table, candidate_union_membership,
#   candidate_raw, off_channel_negative_candidates.
#
# But to make the handoff robust, we explicitly load and validate the
# reusable upstream caches so we can prove that:
#   1) every individual proposer cache is coherent with candidate_raw
#   2) every feature-family cache is coherent with mega_feature_table
# -------------------------------------------------------------------
log("=" * 90)
log("LOADING / VALIDATING UPSTREAM NB03/04 CACHE ARTIFACTS")

detector_cache_dir = None
feature_family_cache_dir = None
if CFG.get("detector_cache_dir_override"):
    detector_cache_dir = Path(CFG["detector_cache_dir_override"])
elif nb34_paths.get("detector_cache_dir"):
    detector_cache_dir = Path(nb34_paths["detector_cache_dir"])

if CFG.get("feature_family_cache_dir_override"):
    feature_family_cache_dir = Path(CFG["feature_family_cache_dir_override"])
elif nb34_paths.get("feature_family_cache_dir"):
    feature_family_cache_dir = Path(nb34_paths["feature_family_cache_dir"])

feature_family_table_paths = {}
if isinstance(nb34_paths.get("feature_family_tables"), dict):
    feature_family_table_paths = {str(k): Path(v) for k, v in nb34_paths["feature_family_tables"].items() if v}

FEATURE_FAMILY_TABLES = {}
DETECTOR_METHOD_TABLES = {}
feature_family_summary_rows = []
detector_method_summary_rows = []
UPSTREAM_CACHE_AUDIT = {
    "detector_cache_dir": str(detector_cache_dir) if detector_cache_dir else None,
    "feature_family_cache_dir": str(feature_family_cache_dir) if feature_family_cache_dir else None,
    "feature_family_cache_loaded": False,
    "detector_method_cache_loaded": False,
    "validated_against_candidate_raw": False,
    "validated_against_mega_feature_table": False,
    "notes": [],
}

# ── Load per-family feature caches from explicit manifest paths ────────────
if CFG.get("load_feature_family_caches", True) and feature_family_table_paths:
    for fam_name, fam_path in sorted(feature_family_table_paths.items()):
        if fam_path.exists():
            fam_df = pd.read_parquet(fam_path)
            FEATURE_FAMILY_TABLES[str(fam_name)] = fam_df
            feature_family_summary_rows.append({
                "feature_family": str(fam_name),
                "path": str(fam_path),
                "n_rows": int(len(fam_df)),
                "n_cols": int(len(fam_df.columns)),
                "n_unique_union_id": int(fam_df["union_id"].astype(str).nunique()) if "union_id" in fam_df.columns else 0,
            })
        else:
            feature_family_summary_rows.append({
                "feature_family": str(fam_name),
                "path": str(fam_path),
                "n_rows": 0,
                "n_cols": 0,
                "n_unique_union_id": 0,
            })
            UPSTREAM_CACHE_AUDIT["notes"].append(f"Missing feature-family cache file: {fam_path}")
    UPSTREAM_CACHE_AUDIT["feature_family_cache_loaded"] = len(FEATURE_FAMILY_TABLES) > 0
    log(f"  feature-family caches      : loaded {len(FEATURE_FAMILY_TABLES)} family tables")
else:
    log("  feature-family caches      : not requested or not advertised by upstream manifest")

# ── Load per-method detector caches from detector_cache_dir ────────────────
if CFG.get("load_detector_method_caches", True) and detector_cache_dir is not None and detector_cache_dir.exists():
    method_cache_paths = sorted(detector_cache_dir.glob("method_candidates__*.parquet"))
    for p in method_cache_paths:
        try:
            dfm = pd.read_parquet(p)
        except Exception as exc:
            detector_method_summary_rows.append({
                "method_id": p.stem.replace("method_candidates__", ""),
                "path": str(p),
                "n_rows": 0,
                "n_cols": 0,
                "status": f"load_error:{type(exc).__name__}",
            })
            if CFG.get("strict_upstream_cache_validation", False):
                raise
            continue
        if "method_id" not in dfm.columns:
            detector_method_summary_rows.append({
                "method_id": p.stem.replace("method_candidates__", ""),
                "path": str(p),
                "n_rows": int(len(dfm)),
                "n_cols": int(len(dfm.columns)),
                "status": "missing_method_id_column",
            })
            if CFG.get("strict_upstream_cache_validation", False):
                raise AssertionError(f"Detector cache {p.name} missing method_id")
            continue
        mids = sorted({str(x) for x in dfm["method_id"].astype(str).unique()})
        assert len(mids) == 1, f"Detector cache {p.name} should contain exactly one method_id, got {mids}"
        mid = mids[0]
        DETECTOR_METHOD_TABLES[mid] = dfm
        detector_method_summary_rows.append({
            "method_id": mid,
            "path": str(p),
            "n_rows": int(len(dfm)),
            "n_cols": int(len(dfm.columns)),
            "status": "ok",
        })
    UPSTREAM_CACHE_AUDIT["detector_method_cache_loaded"] = len(DETECTOR_METHOD_TABLES) > 0
    log(f"  detector method caches     : loaded {len(DETECTOR_METHOD_TABLES)} per-method tables")
else:
    log("  detector method caches     : not requested or directory missing")

FEATURE_FAMILY_SUMMARY = pd.DataFrame(feature_family_summary_rows)
DETECTOR_METHOD_SUMMARY = pd.DataFrame(detector_method_summary_rows)

# ── Validate detector method caches against canonical candidate_raw ────────
if len(candidate_raw) and len(DETECTOR_METHOD_TABLES):
    candidate_raw = candidate_raw.copy()
    candidate_raw["method_id"] = candidate_raw["method_id"].astype(str)

    raw_counts = candidate_raw.groupby("method_id").size().rename("candidate_raw_rows")
    cache_counts = pd.Series({mid: len(dfm) for mid, dfm in DETECTOR_METHOD_TABLES.items()}, name="method_cache_rows")
    detector_counts = pd.concat([raw_counts, cache_counts], axis=1).fillna(0).astype(int).reset_index().rename(columns={"index": "method_id"})
    detector_counts["counts_match"] = detector_counts["candidate_raw_rows"] == detector_counts["method_cache_rows"]

    for _, row in detector_counts.iterrows():
        if not bool(row["counts_match"]):
            msg = f"Detector cache count mismatch for {row['method_id']}: candidate_raw={row['candidate_raw_rows']} cache={row['method_cache_rows']}"
            if CFG.get("strict_upstream_cache_validation", False):
                raise AssertionError(msg)
            UPSTREAM_CACHE_AUDIT["notes"].append(msg)

    common_cmp_cols = [c for c in [
        "crop_id", "dataset_id", "well_id", "method_id",
        "y_px", "x_px", "well_y_px", "well_x_px",
        "score", "panel_key", "wl"
    ] if c in candidate_raw.columns]
    cmp_keys = [c for c in ["method_id", "crop_id", "well_y_px", "well_x_px", "y_px", "x_px"] if c in common_cmp_cols]
    if len(cmp_keys) == 0:
        cmp_keys = [c for c in ["method_id", "crop_id"] if c in common_cmp_cols]

    detector_compare_rows = []
    for mid, cache_df in sorted(DETECTOR_METHOD_TABLES.items()):
        raw_df = candidate_raw[candidate_raw["method_id"] == mid].copy()
        cache_df = cache_df.copy()
        for c in common_cmp_cols:
            if c in raw_df.columns:  raw_df[c]  = raw_df[c].astype(str) if raw_df[c].dtype == object else raw_df[c]
            if c in cache_df.columns: cache_df[c] = cache_df[c].astype(str) if cache_df[c].dtype == object else cache_df[c]

        shared = [c for c in common_cmp_cols if c in raw_df.columns and c in cache_df.columns]
        raw_cmp = raw_df[shared].sort_values(cmp_keys, kind="mergesort").reset_index(drop=True)
        cache_cmp = cache_df[shared].sort_values(cmp_keys, kind="mergesort").reset_index(drop=True)
        same = raw_cmp.equals(cache_cmp)
        detector_compare_rows.append({
            "method_id": mid,
            "candidate_raw_rows": int(len(raw_df)),
            "method_cache_rows": int(len(cache_df)),
            "shared_compare_cols": ",".join(shared),
            "exact_shared_match": bool(same),
        })
        if (not same) and CFG.get("strict_upstream_cache_validation", False):
            raise AssertionError(f"Per-method detector cache mismatch for {mid} on shared columns")
    DETECTOR_CACHE_AUDIT = pd.DataFrame(detector_compare_rows)
    UPSTREAM_CACHE_AUDIT["validated_against_candidate_raw"] = True
    log("  detector cache validation  : candidate_raw parity check passed")
else:
    DETECTOR_CACHE_AUDIT = pd.DataFrame(columns=["method_id","candidate_raw_rows","method_cache_rows","shared_compare_cols","exact_shared_match"])

# ── Validate feature-family caches against canonical mega_feature_table ─────
if len(mega_feature_table) and len(FEATURE_FAMILY_TABLES):
    mega_feature_table = mega_feature_table.copy()
    mega_feature_table["union_id"] = mega_feature_table["union_id"].astype(str)

    meta_cols = {
        "union_id", "dataset_id", "well_id", "crop_id",
        "run_id", "nb04_run_id", "project_config_version",
        "feature_registry_version", "created_at_utc"
    }
    mega_feature_cols = [c for c in mega_feature_table.columns if c not in meta_cols]

    fam_audit_rows = []
    seen_feature_cols = set()
    for fam_name, fam_df in sorted(FEATURE_FAMILY_TABLES.items()):
        fam_df = fam_df.copy()
        assert "union_id" in fam_df.columns, f"Feature-family table {fam_name} missing union_id"
        fam_df["union_id"] = fam_df["union_id"].astype(str)
        assert fam_df["union_id"].is_unique, f"Feature-family table {fam_name} must have unique union_id"
        fam_feature_cols = [c for c in fam_df.columns if c != "union_id"]
        seen_feature_cols.update(fam_feature_cols)

        merged = mega_feature_table[["union_id"] + [c for c in fam_feature_cols if c in mega_feature_table.columns]].merge(
            fam_df[["union_id"] + fam_feature_cols], on="union_id", how="inner", suffixes=("_mega", "_fam")
        )
        shared_cols = [c for c in fam_feature_cols if c in mega_feature_table.columns]
        exact_match = True
        for c in shared_cols:
            left = merged[f"{c}_mega"]
            right = merged[f"{c}_fam"]
            if pd.api.types.is_numeric_dtype(left) and pd.api.types.is_numeric_dtype(right):
                same = np.allclose(left.to_numpy(), right.to_numpy(), equal_nan=True)
            else:
                same = left.fillna("__NA__").astype(str).equals(right.fillna("__NA__").astype(str))
            if not same:
                exact_match = False
                if CFG.get("strict_upstream_cache_validation", False):
                    raise AssertionError(f"Feature-family cache mismatch for {fam_name}:{c}")
                break

        fam_audit_rows.append({
            "feature_family": fam_name,
            "n_rows": int(len(fam_df)),
            "n_feature_cols": int(len(fam_feature_cols)),
            "shared_with_mega": int(len(shared_cols)),
            "all_union_ids_present_in_mega": bool(set(fam_df["union_id"]).issubset(set(mega_feature_table["union_id"]))),
            "exact_shared_match": bool(exact_match),
        })

    missing_from_family_cache = sorted(set(mega_feature_cols) - seen_feature_cols)
    FEATURE_FAMILY_AUDIT = pd.DataFrame(fam_audit_rows)
    UPSTREAM_CACHE_AUDIT["validated_against_mega_feature_table"] = True
    if missing_from_family_cache:
        msg = f"Mega feature table columns not represented in per-family caches: {missing_from_family_cache[:20]}"
        if CFG.get("strict_upstream_cache_validation", False):
            raise AssertionError(msg)
        UPSTREAM_CACHE_AUDIT["notes"].append(msg)
    log("  feature-family validation  : mega_feature_table parity check passed")
else:
    FEATURE_FAMILY_AUDIT = pd.DataFrame(columns=["feature_family","n_rows","n_feature_cols","shared_with_mega","all_union_ids_present_in_mega","exact_shared_match"])

# ── Membership coherence summary ────────────────────────────────────────────
if len(candidate_union_membership):
    candidate_union_membership = candidate_union_membership.copy()
    if "method_id" in candidate_union_membership.columns:
        candidate_union_membership["method_id"] = candidate_union_membership["method_id"].astype(str)
        _membership_method_counts = (
            candidate_union_membership.groupby("method_id").size().rename("membership_rows").reset_index()
        )
    else:
        _membership_method_counts = pd.DataFrame(columns=["method_id", "membership_rows"])
else:
    _membership_method_counts = pd.DataFrame(columns=["method_id", "membership_rows"])

if len(DETECTOR_METHOD_SUMMARY):
    DETECTOR_METHOD_SUMMARY = DETECTOR_METHOD_SUMMARY.merge(
        _membership_method_counts, on="method_id", how="left"
    )
    DETECTOR_METHOD_SUMMARY["membership_rows"] = DETECTOR_METHOD_SUMMARY["membership_rows"].fillna(0).astype(int)
# ensure INPUT_PATHS exists even in fresh kernel runs
if "INPUT_PATHS" not in globals():
    INPUT_PATHS = {}
INPUT_PATHS["detector_cache_dir"] = str(detector_cache_dir) if detector_cache_dir else None
INPUT_PATHS["feature_family_cache_dir"] = str(feature_family_cache_dir) if feature_family_cache_dir else None
INPUT_PATHS["feature_family_tables"] = {k: str(v) for k, v in feature_family_table_paths.items()}

log("Upstream cache handling complete.")


[2026-01-24 04:26:58 UTC] ==========================================================================================
[2026-01-24 04:26:58 UTC] LOADING / VALIDATING UPSTREAM NB03/04 CACHE ARTIFACTS
[2026-01-24 04:26:58 UTC]   feature-family caches      : loaded 11 family tables
[2026-01-24 04:26:58 UTC]   detector method caches     : loaded 18 per-method tables
[2026-01-24 04:26:58 UTC]   detector cache validation  : candidate_raw parity check passed
[2026-01-24 04:26:58 UTC]   feature-family validation  : mega_feature_table parity check passed
[2026-01-24 04:26:58 UTC] Upstream cache handling complete.


/var/folders/mf/blf3nbg52djf7wbt4080gd040000gn/T/ipykernel_22782/2385114811.py:256: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  DETECTOR_METHOD_SUMMARY["membership_rows"] = DETECTOR_METHOD_SUMMARY["membership_rows"].fillna(0).astype(int)


In [19]:

# -------------------------------------------------------------------
# LOAD / VALIDATE NEW OFFTRAIN + OFFAWARE NB03/04 CACHE ARTIFACTS
#
# Appended NB03/04 cells now add:
#   - OFFTRAIN detector universe + per-method detector caches
#   - OFFTRAIN feature-family caches
#   - OFF-aware augmentation tables keyed by deployment union_id
#
# NB05 should:
#   1) load and validate these caches,
#   2) join the deployment-keyed OFF-aware augmentation table into the
#      supervision table for downstream NB06 model training,
#   3) persist audit tables / manifest pointers so NB06 and NB07 can
#      consume the same cache contract without recomputation.
# -------------------------------------------------------------------
log("=" * 90)
log("LOADING / VALIDATING OFFTRAIN + OFFAWARE NB03/04 CACHE ARTIFACTS")

OFFTRAIN_DETECTOR_CACHE_DIR = None
OFFTRAIN_FEATURE_CACHE_DIR = None
OFFAUG_CACHE_DIR = None

if CFG.get("offtrain_detector_cache_dir_override"):
    OFFTRAIN_DETECTOR_CACHE_DIR = Path(CFG["offtrain_detector_cache_dir_override"])
elif detector_cache_dir is not None:
    cand = Path(detector_cache_dir) / "off_training_only"
    if cand.exists():
        OFFTRAIN_DETECTOR_CACHE_DIR = cand

if CFG.get("offtrain_feature_cache_dir_override"):
    OFFTRAIN_FEATURE_CACHE_DIR = Path(CFG["offtrain_feature_cache_dir_override"])
elif feature_family_cache_dir is not None:
    cand = Path(feature_family_cache_dir) / "off_training_only"
    if cand.exists():
        OFFTRAIN_FEATURE_CACHE_DIR = cand

if CFG.get("offaware_aug_cache_dir_override"):
    OFFAUG_CACHE_DIR = Path(CFG["offaware_aug_cache_dir_override"])
elif feature_family_cache_dir is not None:
    cand = Path(feature_family_cache_dir) / "offaware_aug_training_only"
    if cand.exists():
        OFFAUG_CACHE_DIR = cand

OFFTRAIN_TABLES = {}
OFFTRAIN_METHOD_TABLES = {}
OFFTRAIN_FEATURE_TABLES = {}
OFFAUG_TABLES = {}

offtrain_summary_rows = []
offtrain_method_summary_rows = []
offtrain_feature_summary_rows = []
offaug_summary_rows = []

OFFTRAIN_CACHE_AUDIT = {
    "offtrain_detector_cache_dir": str(OFFTRAIN_DETECTOR_CACHE_DIR) if OFFTRAIN_DETECTOR_CACHE_DIR else None,
    "offtrain_feature_cache_dir": str(OFFTRAIN_FEATURE_CACHE_DIR) if OFFTRAIN_FEATURE_CACHE_DIR else None,
    "offaware_aug_cache_dir": str(OFFAUG_CACHE_DIR) if OFFAUG_CACHE_DIR else None,
    "loaded_offtrain_detector_tables": False,
    "loaded_offtrain_feature_tables": False,
    "loaded_offaware_aug_tables": False,
    "validated_offtrain_detector_tables": False,
    "validated_offtrain_feature_tables": False,
    "validated_offaware_aug_tables": False,
    "notes": [],
}

# ── OFFTRAIN detector tables ──────────────────────────────────────────────
if CFG.get("load_offtrain_detector_caches", True) and OFFTRAIN_DETECTOR_CACHE_DIR is not None and OFFTRAIN_DETECTOR_CACHE_DIR.exists():
    for nm in ["candidate_raw", "candidate_union", "candidate_union_membership", "candidate_clusters", "detector_run_summary"]:
        p = OFFTRAIN_DETECTOR_CACHE_DIR / f"{nm}.parquet"
        if p.exists():
            OFFTRAIN_TABLES[nm] = pd.read_parquet(p)
            offtrain_summary_rows.append({
                "table_group": "offtrain_detector",
                "table_name": nm,
                "path": str(p),
                "n_rows": int(len(OFFTRAIN_TABLES[nm])),
                "n_cols": int(len(OFFTRAIN_TABLES[nm].columns)),
                "status": "ok",
            })
        else:
            offtrain_summary_rows.append({
                "table_group": "offtrain_detector",
                "table_name": nm,
                "path": str(p),
                "n_rows": 0,
                "n_cols": 0,
                "status": "missing",
            })
    for p in sorted(OFFTRAIN_DETECTOR_CACHE_DIR.glob("method_candidates__*.parquet")):
        try:
            dfm = pd.read_parquet(p)
            mid = str(dfm["method_id"].astype(str).iloc[0]) if ("method_id" in dfm.columns and len(dfm)) else p.stem.replace("method_candidates__", "")
            OFFTRAIN_METHOD_TABLES[mid] = dfm
            offtrain_method_summary_rows.append({
                "method_id": mid,
                "path": str(p),
                "n_rows": int(len(dfm)),
                "n_cols": int(len(dfm.columns)),
                "status": "ok",
            })
        except Exception as exc:
            offtrain_method_summary_rows.append({
                "method_id": p.stem.replace("method_candidates__", ""),
                "path": str(p),
                "n_rows": 0,
                "n_cols": 0,
                "status": f"load_error:{type(exc).__name__}",
            })
            if CFG.get("strict_upstream_cache_validation", False):
                raise
    OFFTRAIN_CACHE_AUDIT["loaded_offtrain_detector_tables"] = bool(OFFTRAIN_TABLES) or bool(OFFTRAIN_METHOD_TABLES)
    log(f"  offtrain detector caches   : loaded tables={len(OFFTRAIN_TABLES)} methods={len(OFFTRAIN_METHOD_TABLES)}")
else:
    log("  offtrain detector caches   : not requested or directory missing")

offtrain_candidate_raw = OFFTRAIN_TABLES.get("candidate_raw", pd.DataFrame())
offtrain_candidate_union = OFFTRAIN_TABLES.get("candidate_union", pd.DataFrame())
offtrain_candidate_union_membership = OFFTRAIN_TABLES.get("candidate_union_membership", pd.DataFrame())
offtrain_candidate_clusters = OFFTRAIN_TABLES.get("candidate_clusters", pd.DataFrame())
offtrain_detector_run_summary = OFFTRAIN_TABLES.get("detector_run_summary", pd.DataFrame())

OFFTRAIN_SUMMARY = pd.DataFrame(offtrain_summary_rows)
OFFTRAIN_METHOD_SUMMARY = pd.DataFrame(offtrain_method_summary_rows)

if len(offtrain_candidate_raw) and len(OFFTRAIN_METHOD_TABLES):
    _otr = offtrain_candidate_raw.copy()
    _otr["method_id"] = _otr["method_id"].astype(str)
    raw_counts = _otr.groupby("method_id").size().rename("offtrain_candidate_raw_rows")
    cache_counts = pd.Series({mid: len(dfm) for mid, dfm in OFFTRAIN_METHOD_TABLES.items()}, name="offtrain_method_cache_rows")
    cnt = pd.concat([raw_counts, cache_counts], axis=1).fillna(0).astype(int).reset_index().rename(columns={"index": "method_id"})
    cnt["counts_match"] = cnt["offtrain_candidate_raw_rows"] == cnt["offtrain_method_cache_rows"]

    cmp_rows = []
    common_base = [c for c in [
        "crop_id", "dataset_id", "well_id", "method_id",
        "y_px", "x_px", "well_y_px", "well_x_px",
        "score", "score_type", "source_view_id"
    ] if c in _otr.columns]
    cmp_keys = [c for c in ["method_id", "crop_id", "well_y_px", "well_x_px", "y_px", "x_px"] if c in common_base]
    if not cmp_keys:
        cmp_keys = [c for c in ["method_id", "crop_id"] if c in common_base]

    for _, row in cnt.iterrows():
        if not bool(row["counts_match"]):
            msg = f"OFFTRAIN detector cache count mismatch for {row['method_id']}: raw={row['offtrain_candidate_raw_rows']} cache={row['offtrain_method_cache_rows']}"
            if CFG.get("strict_upstream_cache_validation", False):
                raise AssertionError(msg)
            OFFTRAIN_CACHE_AUDIT["notes"].append(msg)

    for mid, cache_df in sorted(OFFTRAIN_METHOD_TABLES.items()):
        raw_df = _otr[_otr["method_id"] == mid].copy()
        cache_df = cache_df.copy()
        shared = [c for c in common_base if c in raw_df.columns and c in cache_df.columns]
        if shared:
            raw_cmp = raw_df[shared].sort_values(cmp_keys, kind="mergesort").reset_index(drop=True)
            cache_cmp = cache_df[shared].sort_values(cmp_keys, kind="mergesort").reset_index(drop=True)
            same = raw_cmp.equals(cache_cmp)
        else:
            same = len(raw_df) == len(cache_df)
        cmp_rows.append({
            "method_id": mid,
            "offtrain_candidate_raw_rows": int(len(raw_df)),
            "offtrain_method_cache_rows": int(len(cache_df)),
            "shared_compare_cols": ",".join(shared),
            "exact_shared_match": bool(same),
        })
        if (not same) and CFG.get("strict_upstream_cache_validation", False):
            raise AssertionError(f"OFFTRAIN per-method detector cache mismatch for {mid}")

    OFFTRAIN_DETECTOR_CACHE_AUDIT = pd.DataFrame(cmp_rows)
    OFFTRAIN_CACHE_AUDIT["validated_offtrain_detector_tables"] = True
    log("  offtrain detector audit    : candidate_raw parity check passed")
else:
    OFFTRAIN_DETECTOR_CACHE_AUDIT = pd.DataFrame(columns=[
        "method_id", "offtrain_candidate_raw_rows", "offtrain_method_cache_rows",
        "shared_compare_cols", "exact_shared_match"
    ])

# ── OFFTRAIN feature tables ───────────────────────────────────────────────
if CFG.get("load_offtrain_feature_caches", True) and OFFTRAIN_FEATURE_CACHE_DIR is not None and OFFTRAIN_FEATURE_CACHE_DIR.exists():
    for nm in ["offf1_core_local", "offf4_proposer_topology", "offf5_barcode_coherence", "offf9_raw_evidence", "offtrain_feature_table"]:
        p = OFFTRAIN_FEATURE_CACHE_DIR / f"{nm}.parquet"
        if p.exists():
            dff = pd.read_parquet(p)
            OFFTRAIN_FEATURE_TABLES[nm] = dff
            offtrain_feature_summary_rows.append({
                "feature_family": nm,
                "path": str(p),
                "n_rows": int(len(dff)),
                "n_cols": int(len(dff.columns)),
                "n_unique_union_id": int(dff["union_id"].astype(str).nunique()) if "union_id" in dff.columns else 0,
                "status": "ok",
            })
        else:
            offtrain_feature_summary_rows.append({
                "feature_family": nm,
                "path": str(p),
                "n_rows": 0,
                "n_cols": 0,
                "n_unique_union_id": 0,
                "status": "missing",
            })
    OFFTRAIN_CACHE_AUDIT["loaded_offtrain_feature_tables"] = len(OFFTRAIN_FEATURE_TABLES) > 0
    log(f"  offtrain feature caches    : loaded {len(OFFTRAIN_FEATURE_TABLES)} tables")
else:
    log("  offtrain feature caches    : not requested or directory missing")

OFFTRAIN_FEATURE_SUMMARY = pd.DataFrame(offtrain_feature_summary_rows)

if len(OFFTRAIN_FEATURE_TABLES):
    fam_rows = []
    off_union_ids = set(offtrain_candidate_union["union_id"].astype(str)) if len(offtrain_candidate_union) and "union_id" in offtrain_candidate_union.columns else set()
    full_tbl = OFFTRAIN_FEATURE_TABLES.get("offtrain_feature_table", pd.DataFrame()).copy()
    if len(full_tbl):
        assert "union_id" in full_tbl.columns, "offtrain_feature_table missing union_id"
        full_tbl["union_id"] = full_tbl["union_id"].astype(str)
        if off_union_ids:
            missing = sorted(set(full_tbl["union_id"]) - off_union_ids)
            if missing and CFG.get("strict_upstream_cache_validation", False):
                raise AssertionError(f"offtrain_feature_table has union_ids not present in offtrain_candidate_union: {missing[:10]}")
    for fam_name, fam_df in sorted(OFFTRAIN_FEATURE_TABLES.items()):
        fam_df = fam_df.copy()
        assert "union_id" in fam_df.columns, f"{fam_name} missing union_id"
        fam_df["union_id"] = fam_df["union_id"].astype(str)
        assert fam_df["union_id"].is_unique, f"{fam_name} must have unique union_id"
        subset_ok = (not off_union_ids) or set(fam_df["union_id"]).issubset(off_union_ids)
        if (not subset_ok) and CFG.get("strict_upstream_cache_validation", False):
            raise AssertionError(f"{fam_name} has union_ids not present in offtrain_candidate_union")
        exact_shared_match = True
        if fam_name != "offtrain_feature_table" and len(full_tbl):
            shared_cols = [c for c in fam_df.columns if c != "union_id" and c in full_tbl.columns]
            merged = full_tbl[["union_id"] + shared_cols].merge(
                fam_df[["union_id"] + shared_cols], on="union_id", how="inner", suffixes=("_full", "_fam")
            )
            for c in shared_cols:
                left = merged[f"{c}_full"]; right = merged[f"{c}_fam"]
                if pd.api.types.is_numeric_dtype(left) and pd.api.types.is_numeric_dtype(right):
                    same = np.allclose(left.to_numpy(), right.to_numpy(), equal_nan=True)
                else:
                    same = left.fillna("__NA__").astype(str).equals(right.fillna("__NA__").astype(str))
                if not same:
                    exact_shared_match = False
                    if CFG.get("strict_upstream_cache_validation", False):
                        raise AssertionError(f"OFFTRAIN feature mismatch for {fam_name}:{c}")
                    break
        fam_rows.append({
            "feature_family": fam_name,
            "n_rows": int(len(fam_df)),
            "n_feature_cols": int(len([c for c in fam_df.columns if c != "union_id"])),
            "all_union_ids_present_in_offtrain_union": bool(subset_ok),
            "exact_shared_match": bool(exact_shared_match),
        })
    OFFTRAIN_FEATURE_AUDIT = pd.DataFrame(fam_rows)
    OFFTRAIN_CACHE_AUDIT["validated_offtrain_feature_tables"] = True
    log("  offtrain feature audit     : per-family parity check passed")
else:
    OFFTRAIN_FEATURE_AUDIT = pd.DataFrame(columns=[
        "feature_family", "n_rows", "n_feature_cols",
        "all_union_ids_present_in_offtrain_union", "exact_shared_match"
    ])

# ── OFF-aware augmentation tables keyed by deployment union_id ────────────
if CFG.get("load_offaware_aug_caches", True) and OFFAUG_CACHE_DIR is not None and OFFAUG_CACHE_DIR.exists():
    for nm in ["f4_offaware_aug", "f5_offaware_aug", "f9_offaware_aug", "f459_offaware_aug"]:
        p = OFFAUG_CACHE_DIR / f"{nm}.parquet"
        if p.exists():
            dfa = pd.read_parquet(p)
            OFFAUG_TABLES[nm] = dfa
            offaug_summary_rows.append({
                "table_name": nm,
                "path": str(p),
                "n_rows": int(len(dfa)),
                "n_cols": int(len(dfa.columns)),
                "n_unique_union_id": int(dfa["union_id"].astype(str).nunique()) if "union_id" in dfa.columns else 0,
                "status": "ok",
            })
        else:
            offaug_summary_rows.append({
                "table_name": nm,
                "path": str(p),
                "n_rows": 0,
                "n_cols": 0,
                "n_unique_union_id": 0,
                "status": "missing",
            })
    OFFTRAIN_CACHE_AUDIT["loaded_offaware_aug_tables"] = len(OFFAUG_TABLES) > 0
    log(f"  off-aware aug caches       : loaded {len(OFFAUG_TABLES)} tables")
else:
    log("  off-aware aug caches       : not requested or directory missing")

OFFAUG_SUMMARY = pd.DataFrame(offaug_summary_rows)

f459_offaware_aug = OFFAUG_TABLES.get("f459_offaware_aug", pd.DataFrame()).copy()
if len(OFFAUG_TABLES):
    dep_union_ids = set(candidate_union["union_id"].astype(str)) if len(candidate_union) and "union_id" in candidate_union.columns else set()
    aug_rows = []
    full_aug = OFFAUG_TABLES.get("f459_offaware_aug", pd.DataFrame()).copy()
    if len(full_aug):
        assert "union_id" in full_aug.columns, "f459_offaware_aug missing union_id"
        full_aug["union_id"] = full_aug["union_id"].astype(str)
        assert full_aug["union_id"].is_unique, "f459_offaware_aug must have unique union_id"
        subset_ok = (not dep_union_ids) or set(full_aug["union_id"]).issubset(dep_union_ids)
        if dep_union_ids and set(full_aug["union_id"]) != dep_union_ids:
            msg = f"f459_offaware_aug union coverage mismatch: aug={len(set(full_aug['union_id']))} deploy_union={len(dep_union_ids)}"
            if CFG.get("strict_upstream_cache_validation", False):
                raise AssertionError(msg)
            OFFTRAIN_CACHE_AUDIT["notes"].append(msg)
    for nm, dfa in sorted(OFFAUG_TABLES.items()):
        dfa = dfa.copy()
        assert "union_id" in dfa.columns, f"{nm} missing union_id"
        dfa["union_id"] = dfa["union_id"].astype(str)
        assert dfa["union_id"].is_unique, f"{nm} must have unique union_id"
        subset_ok = (not dep_union_ids) or set(dfa["union_id"]).issubset(dep_union_ids)
        if (not subset_ok) and CFG.get("strict_upstream_cache_validation", False):
            raise AssertionError(f"{nm} has union_ids not present in candidate_union")
        exact_shared_match = True
        if nm != "f459_offaware_aug" and len(full_aug):
            shared_cols = [c for c in dfa.columns if c != "union_id" and c in full_aug.columns]
            merged = full_aug[["union_id"] + shared_cols].merge(
                dfa[["union_id"] + shared_cols], on="union_id", how="inner", suffixes=("_full", "_part")
            )
            for c in shared_cols:
                left = merged[f"{c}_full"]; right = merged[f"{c}_part"]
                if pd.api.types.is_numeric_dtype(left) and pd.api.types.is_numeric_dtype(right):
                    same = np.allclose(left.to_numpy(), right.to_numpy(), equal_nan=True)
                else:
                    same = left.fillna("__NA__").astype(str).equals(right.fillna("__NA__").astype(str))
                if not same:
                    exact_shared_match = False
                    if CFG.get("strict_upstream_cache_validation", False):
                        raise AssertionError(f"OFF-aware augmentation mismatch for {nm}:{c}")
                    break
        aug_rows.append({
            "table_name": nm,
            "n_rows": int(len(dfa)),
            "n_feature_cols": int(len([c for c in dfa.columns if c != "union_id"])),
            "all_union_ids_present_in_candidate_union": bool(subset_ok),
            "exact_shared_match": bool(exact_shared_match),
        })
    OFFAUG_AUDIT = pd.DataFrame(aug_rows)
    OFFTRAIN_CACHE_AUDIT["validated_offaware_aug_tables"] = True
    log("  off-aware aug audit        : deployment-union parity check passed")
else:
    OFFAUG_AUDIT = pd.DataFrame(columns=[
        "table_name", "n_rows", "n_feature_cols",
        "all_union_ids_present_in_candidate_union", "exact_shared_match"
    ])

INPUT_PATHS["offtrain_detector_cache_dir"] = str(OFFTRAIN_DETECTOR_CACHE_DIR) if OFFTRAIN_DETECTOR_CACHE_DIR else None
INPUT_PATHS["offtrain_feature_cache_dir"] = str(OFFTRAIN_FEATURE_CACHE_DIR) if OFFTRAIN_FEATURE_CACHE_DIR else None
INPUT_PATHS["offaware_aug_cache_dir"] = str(OFFAUG_CACHE_DIR) if OFFAUG_CACHE_DIR else None

UPSTREAM_CACHE_AUDIT["offtrain_cache_audit"] = OFFTRAIN_CACHE_AUDIT
log("OFFTRAIN / OFFAWARE cache handling complete.")


[2026-01-24 04:26:58 UTC] ==========================================================================================
[2026-01-24 04:26:58 UTC] LOADING / VALIDATING OFFTRAIN + OFFAWARE NB03/04 CACHE ARTIFACTS
[2026-01-24 04:26:58 UTC]   offtrain detector caches   : not requested or directory missing
[2026-01-24 04:26:58 UTC]   offtrain feature caches    : not requested or directory missing
[2026-01-24 04:26:58 UTC]   off-aware aug caches       : not requested or directory missing
[2026-01-24 04:26:58 UTC] OFFTRAIN / OFFAWARE cache handling complete.


In [20]:

# -------------------------------------------------------------------
# FULL-WELL → QA-CROP SPATIAL MASKING + METADATA ALIGNMENT
#
# NB03/04 now emits a true full-well candidate universe / feature table.
# Notebook 05 must NOT regenerate candidates/features on crops.
# Instead, it spatially masks the full-well rows into the QA crop windows
# defined by crop_registry, then performs matching/splits on those masked rows.
#
# This preserves exact feature parity with NB07:
#   - bgscale_* and *_global_norm come directly from NB03/04 full-well outputs
#   - no proposer/feature recomputation occurs here
#   - crop_registry is supervision territory only
# -------------------------------------------------------------------
log("=" * 90)
log("FULL-WELL → QA-CROP SPATIAL MASKING")

# Preserve upstream full-well tables for provenance / debugging.
fullwell_candidate_union = candidate_union.copy()
fullwell_mega_feature_table = mega_feature_table.copy()
fullwell_off_channel_negative_candidates = off_channel_negative_candidates.copy()

for _df_name in ["crop_registry", "fullwell_candidate_union", "fullwell_mega_feature_table"]:
    _df = globals()[_df_name]
    for _c in [c for c in ["dataset_id", "well_id", "crop_id"] if c in _df.columns]:
        _df[_c] = _df[_c].astype(str)

required_crop_cols = {
    "dataset_id", "well_id", "crop_id",
    "well_ymin_px", "well_ymax_px", "well_xmin_px", "well_xmax_px",
}
missing_crop_cols = sorted(required_crop_cols - set(crop_registry.columns))
assert not missing_crop_cols, f"crop_registry missing required bbox columns: {missing_crop_cols}"

required_union_xy = {"union_id", "dataset_id", "well_id", "union_medoid_well_y_px", "union_medoid_well_x_px"}
missing_union_xy = sorted(required_union_xy - set(fullwell_candidate_union.columns))
assert not missing_union_xy, f"candidate_union missing required full-well coordinate columns: {missing_union_xy}"

# Ensure feature metadata exists before masking.
_meta_cols_to_add = ["union_id", "dataset_id", "well_id"]
_missing_in_mega = [c for c in _meta_cols_to_add if c not in fullwell_mega_feature_table.columns]
if _missing_in_mega:
    _cu_meta = fullwell_candidate_union[_meta_cols_to_add].drop_duplicates("union_id")
    fullwell_mega_feature_table = fullwell_mega_feature_table.merge(_cu_meta, on="union_id", how="left")
    log(f"  fullwell_mega_feature_table enriched with: {_missing_in_mega}")

# one feature row per union_id expected from NB03/04
fullwell_candidate_union["union_id"] = fullwell_candidate_union["union_id"].astype(str)
fullwell_mega_feature_table["union_id"] = fullwell_mega_feature_table["union_id"].astype(str)
assert fullwell_candidate_union["union_id"].is_unique, "Full-well candidate_union must have unique union_id"
assert fullwell_mega_feature_table["union_id"].is_unique, "Full-well mega_feature_table must have unique union_id"
assert set(fullwell_candidate_union["union_id"]) == set(fullwell_mega_feature_table["union_id"]), (
    "Mismatch between full-well candidate_union and mega_feature_table union_id sets"
)

crop_registry = crop_registry.copy()
for c in ["well_ymin_px", "well_ymax_px", "well_xmin_px", "well_xmax_px"]:
    crop_registry[c] = pd.to_numeric(crop_registry[c], errors="raise")

masked_union_parts = []
crop_mask_summary_rows = []

for i, crow in crop_registry.sort_values(["dataset_id", "well_id", "crop_id"]).reset_index(drop=True).iterrows():
    ds = str(crow["dataset_id"])
    wid = str(crow["well_id"])
    cid = str(crow["crop_id"])
    ymin, ymax = float(crow["well_ymin_px"]), float(crow["well_ymax_px"])
    xmin, xmax = float(crow["well_xmin_px"]), float(crow["well_xmax_px"])

    sub = fullwell_candidate_union[fullwell_candidate_union["well_id"].astype(str) == wid].copy()
    # dataset_id filter skipped: fullwell table may have well_id in dataset_id column

    keep = (
        (pd.to_numeric(sub["union_medoid_well_y_px"], errors="coerce") >= ymin) &
        (pd.to_numeric(sub["union_medoid_well_y_px"], errors="coerce") <  ymax) &
        (pd.to_numeric(sub["union_medoid_well_x_px"], errors="coerce") >= xmin) &
        (pd.to_numeric(sub["union_medoid_well_x_px"], errors="coerce") <  xmax)
    )
    hit = sub.loc[keep].copy()
    hit["dataset_id"] = ds
    hit["well_id"] = wid
    hit["crop_id"] = cid
    masked_union_parts.append(hit)

    crop_mask_summary_rows.append({
        "dataset_id": ds,
        "well_id": wid,
        "crop_id": cid,
        "n_union_candidates_in_crop": int(len(hit)),
        "well_ymin_px": ymin,
        "well_ymax_px": ymax,
        "well_xmin_px": xmin,
        "well_xmax_px": xmax,
    })

candidate_union = (
    pd.concat(masked_union_parts, ignore_index=True)
    if masked_union_parts else
    fullwell_candidate_union.iloc[0:0].copy()
)

# one row per (union_id, crop_id) is allowed because overlapping QA crops may duplicate the same full-well union_id
assert candidate_union[["union_id", "crop_id"]].drop_duplicates().shape[0] == len(candidate_union), (
    "Duplicate (union_id, crop_id) pairs after spatial masking"
)

# Rebuild mega table by attaching the QA crop_id to the full-well feature rows
masked_union_meta = candidate_union[["union_id", "dataset_id", "well_id", "crop_id"]].drop_duplicates()
mega_feature_table = masked_union_meta.merge(
    fullwell_mega_feature_table.drop(columns=[c for c in ["dataset_id", "well_id", "crop_id"] if c in fullwell_mega_feature_table.columns]),
    on="union_id",
    how="left",
    validate="many_to_one",
)

# Spatially mask off-channel negative side table too, without recomputation
if len(fullwell_off_channel_negative_candidates):
    off = fullwell_off_channel_negative_candidates.copy()
    for c in [c for c in ["dataset_id", "well_id", "crop_id"] if c in off.columns]:
        off[c] = off[c].astype(str)
    masked_off_parts = []
    for _, crow in crop_registry.sort_values(["dataset_id", "well_id", "crop_id"]).iterrows():
        ds = str(crow["dataset_id"]); wid = str(crow["well_id"]); cid = str(crow["crop_id"])
        ymin, ymax = float(crow["well_ymin_px"]), float(crow["well_ymax_px"])
        xmin, xmax = float(crow["well_xmin_px"]), float(crow["well_xmax_px"])
        sub = off[off["well_id"].astype(str) == wid].copy()
        # dataset_id filter skipped
        keep = (
            (pd.to_numeric(sub["well_y_px"], errors="coerce") >= ymin) &
            (pd.to_numeric(sub["well_y_px"], errors="coerce") <  ymax) &
            (pd.to_numeric(sub["well_x_px"], errors="coerce") >= xmin) &
            (pd.to_numeric(sub["well_x_px"], errors="coerce") <  xmax)
        )
        hit = sub.loc[keep].copy()
        hit["dataset_id"] = ds
        hit["well_id"] = wid
        hit["crop_id"] = cid
        masked_off_parts.append(hit)
    off_channel_negative_candidates = (
        pd.concat(masked_off_parts, ignore_index=True)
        if masked_off_parts else
        off.iloc[0:0].copy()
    )
else:
    off_channel_negative_candidates = fullwell_off_channel_negative_candidates.copy()

crop_mask_summary = pd.DataFrame(crop_mask_summary_rows)

# Strong alignment checks after masking
candidate_union["union_id"] = candidate_union["union_id"].astype(str)
mega_feature_table["union_id"] = mega_feature_table["union_id"].astype(str)
mega_ids = set(mega_feature_table["union_id"])
cu_ids = set(candidate_union["union_id"])
missing_in_mega = cu_ids - mega_ids
assert len(missing_in_mega) == 0, f"{len(missing_in_mega)} masked union_ids missing from mega_feature_table"

log(f"  full-well candidate_union      : {len(fullwell_candidate_union):,} rows")
log(f"  QA crop windows                : {crop_registry['crop_id'].nunique():,} crops")
log(f"  masked candidate_union         : {len(candidate_union):,} rows")
log(f"  masked mega_feature_table      : {len(mega_feature_table):,} rows")
log(f"  masked off_channel_negatives   : {len(off_channel_negative_candidates):,} rows")
if len(candidate_union) == 0:
    log("[WARN] Spatial masking produced 0 candidates — checking full-well coordinate ranges:")
    log(f"  fullwell candidate_union rows : {len(fullwell_candidate_union)}")
    if len(fullwell_candidate_union):
        log(f"  well_ids in fullwell union    : {fullwell_candidate_union['well_id'].unique().tolist()}")
        log(f"  union_medoid_well_y_px range  : {fullwell_candidate_union['union_medoid_well_y_px'].min():.0f} - {fullwell_candidate_union['union_medoid_well_y_px'].max():.0f}")
        log(f"  union_medoid_well_x_px range  : {fullwell_candidate_union['union_medoid_well_x_px'].min():.0f} - {fullwell_candidate_union['union_medoid_well_x_px'].max():.0f}")
    log(f"  crop_registry bbox y range    : {crop_registry['well_ymin_px'].min()} - {crop_registry['well_ymax_px'].max()}")
    log(f"  crop_registry bbox x range    : {crop_registry['well_xmin_px'].min()} - {crop_registry['well_xmax_px'].max()}")
display(crop_mask_summary.head())


[2026-01-24 04:26:58 UTC] ==========================================================================================
[2026-01-24 04:26:58 UTC] FULL-WELL → QA-CROP SPATIAL MASKING
[2026-01-24 04:26:58 UTC]   fullwell_mega_feature_table enriched with: ['dataset_id', 'well_id']
[2026-01-24 04:26:59 UTC]   full-well candidate_union      : 5,955 rows
[2026-01-24 04:26:59 UTC]   QA crop windows                : 50 crops
[2026-01-24 04:26:59 UTC]   masked candidate_union         : 864 rows
[2026-01-24 04:26:59 UTC]   masked mega_feature_table      : 864 rows
[2026-01-24 04:26:59 UTC]   masked off_channel_negatives   : 2,397 rows


,dataset_id,well_id,crop_id,n_union_candidates_in_crop,well_ymin_px,well_ymax_px,well_xmin_px,well_xmax_px
0,dataset_default,C8,crop_C8_0adf818b2b6b,8,256.0,768.0,4352.0,4864.0
1,dataset_default,C8,crop_C8_1ffcd18f3eed,3,6912.0,7424.0,7168.0,7680.0
2,dataset_default,C8,crop_C8_2b359db36154,3,1792.0,2304.0,5632.0,6144.0
3,dataset_default,C8,crop_C8_2b6411e62d9d,106,0.0,512.0,6656.0,7168.0
4,dataset_default,C8,crop_C8_3f4741bac138,13,1280.0,1792.0,5632.0,6144.0


In [21]:
# -------------------------------------------------------------------
# CONTRACT VALIDATION
# Asserts required columns, uniqueness, dtype normalization, label validity.
# -------------------------------------------------------------------
required_raw_cols = {"union_id", "well_id", "crop_id"}

required_union_cols = {
    "union_id", "dataset_id", "well_id", "crop_id",
    "union_centroid_well_y_px", "union_centroid_well_x_px",
    "union_medoid_well_y_px", "union_medoid_well_x_px",
}
required_mega_cols = {
    "union_id", "dataset_id", "well_id", "crop_id",
}
required_ann_cols = {
    "annotation_id", "dataset_id", "well_id", "crop_id",
    "label", "confidence",
    "raw_click_well_y_px", "raw_click_well_x_px",
}
required_crop_cols = {
    "crop_id", "dataset_id", "well_id",
    "well_ymin_px", "well_xmin_px", "well_ymax_px", "well_xmax_px",
    "annotator_status", "crop_registry_version",
}
required_offneg_cols = {
    "dataset_id", "well_id", "crop_id", "well_y_px", "well_x_px"
}

missing_raw = sorted(required_raw_cols - set(candidate_raw.columns)) if len(candidate_raw) else []
missing_union = sorted(required_union_cols - set(candidate_union.columns))
missing_mega  = sorted(required_mega_cols  - set(mega_feature_table.columns))
missing_ann   = sorted(required_ann_cols   - set(annotations.columns))
missing_crop  = sorted(required_crop_cols  - set(crop_registry.columns))
missing_offneg = sorted(required_offneg_cols - set(off_channel_negative_candidates.columns)) if len(off_channel_negative_candidates) else []

if len(candidate_raw):
    assert not missing_raw, f"candidate_raw missing columns: {missing_raw}"
assert not missing_union, f"candidate_union missing columns: {missing_union}"
assert not missing_mega,  f"mega_feature_table missing columns: {missing_mega}"
assert not missing_ann,   f"annotations missing columns: {missing_ann}"
assert not missing_crop,  f"crop_registry missing columns: {missing_crop}"
if len(off_channel_negative_candidates):
    assert not missing_offneg, f"off_channel_negative_candidates missing columns: {missing_offneg}"

assert candidate_union["union_id"].is_unique,       "candidate_union: union_id must be unique"
assert mega_feature_table["union_id"].is_unique,    "mega_feature_table: union_id must be unique"
assert annotations["annotation_id"].is_unique,      "annotations: annotation_id must be unique"
assert crop_registry["crop_id"].is_unique,          "crop_registry: crop_id must be unique"

# Normalize ID column dtypes to str (guard against int crop_id from parquet vs str from yaml)
for df in [candidate_union, mega_feature_table, annotations, crop_registry]:
    for c in ["dataset_id", "well_id", "crop_id"]:
        if c in df.columns:
            df[c] = df[c].astype(str)

if len(candidate_raw):
    for c in ["dataset_id", "well_id", "crop_id", "union_id"]:
        if c in candidate_raw.columns:
            candidate_raw[c] = candidate_raw[c].astype(str)

if len(off_channel_negative_candidates):
    for c in ["dataset_id", "well_id", "crop_id"]:
        if c in off_channel_negative_candidates.columns:
            off_channel_negative_candidates[c] = off_channel_negative_candidates[c].astype(str)
    if "is_off_channel_negative" in off_channel_negative_candidates.columns:
        off_channel_negative_candidates["is_off_channel_negative"] = (
            off_channel_negative_candidates["is_off_channel_negative"].fillna(True).astype(bool)
        )
    else:
        off_channel_negative_candidates["is_off_channel_negative"] = True

for c in ["well_ymin_px", "well_xmin_px", "well_ymax_px", "well_xmax_px"]:
    crop_registry[c] = crop_registry[c].astype(float)

valid_labels = {"TRUE_SPOT", "EXPLICIT_NEGATIVE"}
actual_labels = set(annotations["label"].astype(str).unique())
unexpected_labels = actual_labels - valid_labels
if unexpected_labels:
    log(f"[warn] Unexpected annotation labels (will be treated as excluded): {unexpected_labels}")

log("Contract validation passed.")


[2026-01-24 04:26:59 UTC] Contract validation passed.


In [22]:
# -------------------------------------------------------------------
# ANNOTATION COORDINATE PREPARATION
# -------------------------------------------------------------------
def choose_truth_well_coords(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    def _to_float(series):
        return pd.to_numeric(series, errors="coerce") if series.name in out.columns else pd.Series(np.nan, index=out.index)

    ry = _to_float(out.get("refined_click_well_y_px", pd.Series(np.nan, index=out.index, name="refined_click_well_y_px")))
    rx = _to_float(out.get("refined_click_well_x_px", pd.Series(np.nan, index=out.index, name="refined_click_well_x_px")))
    wy = _to_float(out.get("raw_click_well_y_px",     pd.Series(np.nan, index=out.index, name="raw_click_well_y_px")))
    wx = _to_float(out.get("raw_click_well_x_px",     pd.Series(np.nan, index=out.index, name="raw_click_well_x_px")))

    out["truth_well_y_px"]    = np.where(np.isfinite(ry), ry, wy)
    out["truth_well_x_px"]    = np.where(np.isfinite(rx), rx, wx)
    out["truth_coord_source"] = np.where(
        np.isfinite(ry) & np.isfinite(rx), "refined_well", "raw_well"
    )
    return out

if annotations is None or len(annotations) == 0:
    annotations = pd.DataFrame(columns=["dataset_id","well_id","crop_id","annotation_id","label","refined_click_well_y_px","refined_click_well_x_px","raw_click_well_y_px","raw_click_well_x_px"])
else:
    annotations = annotations.copy()
    if "label" not in annotations.columns:
        annotations["label"] = pd.Series(dtype="object")
annotations = choose_truth_well_coords(annotations)

bad_truth_rows = annotations[
    ~np.isfinite(annotations["truth_well_y_px"]) |
    ~np.isfinite(annotations["truth_well_x_px"])
]
if len(bad_truth_rows) > 0:
    log(f"[warn] {len(bad_truth_rows)} annotation rows with invalid truth coordinates — skipping.")
    annotations = annotations[
        np.isfinite(annotations["truth_well_y_px"]) &
        np.isfinite(annotations["truth_well_x_px"])
    ].reset_index(drop=True)

# ── DEDUPLICATE annotations ────────────────────────────────────────────────
# The annotations parquet has one row per panel per spot (5× duplicated).
# Deduplicate on (crop_id, label, well_y, well_x) so each physical spot
# appears exactly once. This prevents one candidate matching 5 copies of
# the same spot, which would corrupt union_id uniqueness in matched_positive.
n_before = len(annotations)
annotations = annotations.drop_duplicates(
    subset=["dataset_id", "well_id", "crop_id", "label", "truth_well_y_px", "truth_well_x_px"]
).reset_index(drop=True)
n_after = len(annotations)
log(f"Annotation deduplication: {n_before:,} → {n_after:,} rows (removed {n_before-n_after:,} panel duplicates)")

n_true = int((annotations["label"] == "TRUE_SPOT").sum())
n_neg  = int((annotations["label"] == "EXPLICIT_NEGATIVE").sum())
log(f"Annotations ready: {len(annotations):,} total — {n_true:,} TRUE_SPOT  {n_neg:,} EXPLICIT_NEGATIVE")
log(f"Using refined coords: {(annotations['truth_coord_source'] == 'refined_well').sum():,} | raw fallback: {(annotations['truth_coord_source'] == 'raw_well').sum():,}")
display(annotations[["annotation_id", "well_id", "crop_id", "label", "truth_well_y_px", "truth_well_x_px", "truth_coord_source"]].head())

[2026-01-24 04:26:59 UTC] Annotation deduplication: 1,795 → 359 rows (removed 1,436 panel duplicates)
[2026-01-24 04:26:59 UTC] Annotations ready: 359 total — 303 TRUE_SPOT  56 EXPLICIT_NEGATIVE
[2026-01-24 04:26:59 UTC] Using refined coords: 359 | raw fallback: 0


,annotation_id,well_id,crop_id,label,truth_well_y_px,truth_well_x_px,truth_coord_source
0,ann_907a221691c0c636,C8,crop_C8_0adf818b2b6b,TRUE_SPOT,450.0,4504.0,refined_well
1,ann_9563cd172c3a864f,C8,crop_C8_0adf818b2b6b,TRUE_SPOT,619.0,4512.0,refined_well
2,ann_fe0dc92ca60c7a40,C8,crop_C8_0adf818b2b6b,TRUE_SPOT,464.0,4689.0,refined_well
3,ann_78ad52ae1a9e7f63,C8,crop_C8_1ffcd18f3eed,TRUE_SPOT,7366.0,7190.0,refined_well
4,ann_06f7ca83424571c2,C8,crop_C8_1ffcd18f3eed,TRUE_SPOT,7298.0,7175.0,refined_well


In [23]:
print("candidate_union_path:", candidate_union_path)
print()

import pandas as pd
if candidate_union_path is not None:
    df = pd.read_parquet(candidate_union_path)
    print("rows:", len(df))
    print("columns:", df.columns.tolist()[:10])
    if len(df):
        print(df[["well_id","crop_id","union_medoid_well_y_px","union_medoid_well_x_px"]].head())
    else:
        print("FILE IS EMPTY")

candidate_union_path: None



In [24]:
# -------------------------------------------------------------------
# QA CROP TERRITORY / COMPLETION RESOLUTION
# -------------------------------------------------------------------
ann_counts = (
    annotations
    .groupby(["dataset_id", "well_id", "crop_id", "label"], dropna=False)
    .size().rename("n_annotations").reset_index()
)
ann_counts_wide = ann_counts.pivot_table(
    index=["dataset_id", "well_id", "crop_id"],
    columns="label",
    values="n_annotations",
    aggfunc="sum",
    fill_value=0,
).reset_index()
for col in ["TRUE_SPOT", "EXPLICIT_NEGATIVE"]:
    if col not in ann_counts_wide.columns:
        ann_counts_wide[col] = 0

crop_status = crop_registry.merge(ann_counts_wide, on=["dataset_id", "well_id", "crop_id"], how="left")
crop_status["TRUE_SPOT"]          = crop_status["TRUE_SPOT"].fillna(0).astype(int)
crop_status["EXPLICIT_NEGATIVE"]  = crop_status["EXPLICIT_NEGATIVE"].fillna(0).astype(int)
crop_status["total_annotations"]  = crop_status["TRUE_SPOT"] + crop_status["EXPLICIT_NEGATIVE"]

min_ann = CFG["min_annotations_for_negative_generation"]
crop_status["is_completed_for_negatives"] = crop_status["total_annotations"] >= min_ann

# Idempotent merge onto candidate_union
for col in ["annotator_status", "crop_registry_version", "is_completed_for_negatives", "TRUE_SPOT", "EXPLICIT_NEGATIVE"]:
    if col in candidate_union.columns:
        candidate_union = candidate_union.drop(columns=[col])

candidate_union = candidate_union.merge(
    crop_status[[
        "dataset_id", "well_id", "crop_id",
        "annotator_status", "crop_registry_version",
        "is_completed_for_negatives", "TRUE_SPOT", "EXPLICIT_NEGATIVE"
    ]],
    on=["dataset_id", "well_id", "crop_id"],
    how="left",
)

# ── Resolve side-table off-channel negatives (DO NOT union-tag candidates) ──
OFF_NEG_METHOD = "off_channel_negative_sampler"

if len(off_channel_negative_candidates):
    off_channel_negative_candidates = off_channel_negative_candidates.copy()
    for c in ["dataset_id", "well_id", "crop_id"]:
        if c in off_channel_negative_candidates.columns:
            off_channel_negative_candidates[c] = off_channel_negative_candidates[c].astype(str)

    # Keep only the current candidate universe territory
    valid_crop_keys = set(
        candidate_union[["dataset_id", "well_id", "crop_id"]]
        .drop_duplicates()
        .astype(str)
        .itertuples(index=False, name=None)
    )
    off_channel_negative_candidates["_crop_key"] = list(
        zip(
            off_channel_negative_candidates["dataset_id"].astype(str),
            off_channel_negative_candidates["well_id"].astype(str),
            off_channel_negative_candidates["crop_id"].astype(str),
        )
    )
    off_channel_negative_candidates = off_channel_negative_candidates[
        off_channel_negative_candidates["_crop_key"].isin(valid_crop_keys)
    ].copy()
    off_channel_negative_candidates = off_channel_negative_candidates.drop(columns=["_crop_key"])

    if "raw_detection_id" in off_channel_negative_candidates.columns:
        off_channel_negative_candidates = off_channel_negative_candidates.drop_duplicates("raw_detection_id")
    else:
        off_channel_negative_candidates = off_channel_negative_candidates.drop_duplicates(
            subset=["dataset_id", "well_id", "crop_id", "well_y_px", "well_x_px"]
        )

    if "is_off_channel_negative" not in off_channel_negative_candidates.columns:
        off_channel_negative_candidates["is_off_channel_negative"] = True
    else:
        off_channel_negative_candidates["is_off_channel_negative"] = (
            off_channel_negative_candidates["is_off_channel_negative"].fillna(True).astype(bool)
        )

    log(
        "Resolved off-channel negative side table: "
        f"{len(off_channel_negative_candidates):,} rows across "
        f"{off_channel_negative_candidates['crop_id'].nunique():,} crops"
    )
else:
    off_channel_negative_candidates = pd.DataFrame(columns=[
        "dataset_id", "well_id", "crop_id", "well_y_px", "well_x_px", "is_off_channel_negative"
    ])
    log("Resolved off-channel negative side table: 0 rows")

# Keep candidate_union pure: these are real unionized spot hypotheses only.
if "is_off_channel_negative" in candidate_union.columns:
    candidate_union = candidate_union.drop(columns=["is_off_channel_negative"])

# Idempotent merge onto mega
for col in ["is_completed_for_negatives", "coverage_allows_negative", "eligible_for_negative_supervision"]:
    if col in mega_feature_table.columns:
        mega_feature_table = mega_feature_table.drop(columns=[col])

mega_feature_table = mega_feature_table.merge(
    crop_status[["dataset_id", "well_id", "crop_id", "is_completed_for_negatives"]],
    on=["dataset_id", "well_id", "crop_id"],
    how="left",
)

if "annotation_coverage_status" in mega_feature_table.columns and CFG["use_mega_annotation_coverage_if_available"]:
    ok_cov = set(CFG["accepted_annotation_coverage_statuses_for_negatives"])
    mega_feature_table["coverage_allows_negative"] = (
        mega_feature_table["annotation_coverage_status"].astype(str).isin(ok_cov)
    )
    mega_feature_table["eligible_for_negative_supervision"] = (
        mega_feature_table["coverage_allows_negative"]
        | mega_feature_table["is_completed_for_negatives"].fillna(False)
    )
else:
    mega_feature_table["coverage_allows_negative"] = mega_feature_table["is_completed_for_negatives"].fillna(False)
    mega_feature_table["eligible_for_negative_supervision"] = (
        mega_feature_table["is_completed_for_negatives"].fillna(False)
    )

if "eligible_for_negative_supervision" in candidate_union.columns:
    candidate_union = candidate_union.drop(columns=["eligible_for_negative_supervision"])

candidate_union = candidate_union.merge(
    mega_feature_table[["union_id", "eligible_for_negative_supervision"]].drop_duplicates("union_id"),
    on="union_id",
    how="left",
)

FULLY_ANNOTATED_CROPS = set(
    crop_status.loc[crop_status["is_completed_for_negatives"], "crop_id"].astype(str).unique()
)
log(f"Crops eligible for negative generation: {len(FULLY_ANNOTATED_CROPS)}")

annotated_crop_ids = set(annotations["crop_id"].dropna().astype(str).unique())
candidate_crop_ids = set(candidate_union["crop_id"].dropna().astype(str).unique())
offneg_crop_ids    = set(off_channel_negative_candidates["crop_id"].dropna().astype(str).unique()) if len(off_channel_negative_candidates) else set()
overlap_crops      = annotated_crop_ids & candidate_crop_ids
log(f"Annotated crops: {len(annotated_crop_ids)} | Candidate crops: {len(candidate_crop_ids)} | Overlap: {len(overlap_crops)}")
log(f"Off-neg side-table crops overlapping candidate crops: {len(offneg_crop_ids & candidate_crop_ids)}")

assert len(overlap_crops) > 0, (
    "No overlap between annotated crops and candidate crops. "
    "Check crop_id consistency between NB02 and NB03."
)

display(
    crop_status[[
        "dataset_id", "well_id", "crop_id", "annotator_status",
        "is_completed_for_negatives", "TRUE_SPOT", "EXPLICIT_NEGATIVE", "total_annotations"
    ]].sort_values(["well_id", "crop_id"])
)


[2026-01-24 04:26:59 UTC] Resolved off-channel negative side table: 2,395 rows across 49 crops
[2026-01-24 04:26:59 UTC] Crops eligible for negative generation: 50
[2026-01-24 04:26:59 UTC] Annotated crops: 50 | Candidate crops: 49 | Overlap: 49
[2026-01-24 04:26:59 UTC] Off-neg side-table crops overlapping candidate crops: 49


,dataset_id,well_id,crop_id,annotator_status,is_completed_for_negatives,TRUE_SPOT,EXPLICIT_NEGATIVE,total_annotations
4,dataset_default,C8,crop_C8_0adf818b2b6b,pending,True,3,0,3
23,dataset_default,C8,crop_C8_1ffcd18f3eed,pending,True,2,0,2
12,dataset_default,C8,crop_C8_2b359db36154,pending,True,3,0,3
2,dataset_default,C8,crop_C8_2b6411e62d9d,pending,True,1,3,4
8,dataset_default,C8,crop_C8_3f4741bac138,pending,True,6,0,6
14,dataset_default,C8,crop_C8_495ce548be49,pending,True,0,1,1
3,dataset_default,C8,crop_C8_49b3266fc66f,pending,True,0,5,5
19,dataset_default,C8,crop_C8_4f61d31afbef,pending,True,2,0,2
17,dataset_default,C8,crop_C8_67faf4382509,pending,True,1,0,1
6,dataset_default,C8,crop_C8_70420ab6a23a,pending,True,4,1,5


In [25]:
# -------------------------------------------------------------------
# MATCHING HELPERS
# -------------------------------------------------------------------
def stable_match_id(union_id, annotation_id, match_status, radius, registry_version) -> str:
    raw = f"{union_id}|{annotation_id}|{match_status}|{radius}|{registry_version}"
    return "match_" + hashlib.sha1(raw.encode("utf-8")).hexdigest()[:16]

def euclidean_distance_matrix(cand_yx: np.ndarray, truth_yx: np.ndarray) -> np.ndarray:
    if len(cand_yx) == 0 or len(truth_yx) == 0:
        return np.empty((len(cand_yx), len(truth_yx)), dtype=np.float32)
    dy = cand_yx[:, None, 0] - truth_yx[None, :, 0]
    dx = cand_yx[:, None, 1] - truth_yx[None, :, 1]
    return np.sqrt(dy * dy + dx * dx).astype(np.float32)

def thresholded_hungarian(cand_yx: np.ndarray, truth_yx: np.ndarray, radius: float):
    """Returns list of (cand_idx, truth_idx, distance_px) for matched pairs."""
    D = euclidean_distance_matrix(cand_yx, truth_yx)
    if D.size == 0:
        return [], D
    BIG = 1e9
    C = D.copy()
    C[C > radius] = BIG
    rows, cols = linear_sum_assignment(C)
    kept = [(int(r), int(c), float(D[r, c])) for r, c in zip(rows, cols) if C[r, c] < BIG]
    return kept, D

def halo_exclude_candidates(cand_yx: np.ndarray, uncertain_yx: np.ndarray, radius: float) -> set:
    """
    Returns the set of candidate indices that fall within `radius` px
    of ANY uncertain annotation (halo exclusion policy).
    """
    if len(cand_yx) == 0 or len(uncertain_yx) == 0:
        return set()
    tree = cKDTree(uncertain_yx)
    excluded = set()
    for i, cy in enumerate(cand_yx):
        idxs = tree.query_ball_point(cy, r=radius)
        if idxs:
            excluded.add(i)
    return excluded

def nearest_truth_distance(cand_yx: np.ndarray, truth_yx: np.ndarray) -> np.ndarray:
    if len(cand_yx) == 0:
        return np.array([], dtype=np.float32)
    if len(truth_yx) == 0:
        return np.full(len(cand_yx), np.inf, dtype=np.float32)
    tree = cKDTree(truth_yx)
    dist, _ = tree.query(cand_yx, k=1)
    return np.asarray(dist, dtype=np.float32)

In [28]:
# -------------------------------------------------------------------
# PER-QA-CROP HUNGARIAN MATCHING ON MASKED FULL-WELL CANDIDATES  (§9.1)
# -------------------------------------------------------------------
log("=" * 90)
log("BEGIN MATCHING")
t0_match = time.perf_counter()

MATCH_RADIUS        = float(CFG["truth_match_radius_px"])
EXPLICIT_NEG_RADIUS = float(CFG["explicit_negative_match_radius_px"])
MATCH_VERSION       = CFG["matching_registry_version"]

MATCH_ROWS         = []
TRUTH_SUMMARY_ROWS = []
MATCH_TIMINGS      = []
CROP_SUMMARY_ROWS  = []

crop_union_groups = {k: g for k, g in candidate_union.groupby(["dataset_id", "well_id", "crop_id"], sort=True)}
crop_ann_groups   = {k: g for k, g in annotations.groupby(["dataset_id", "well_id", "crop_id"], sort=True)}
crop_offneg_groups = (
    {k: g for k, g in off_channel_negative_candidates.groupby(["dataset_id", "well_id", "crop_id"], sort=True)}
    if len(off_channel_negative_candidates) else {}
)

all_crop_keys = sorted(set(crop_union_groups.keys()) | set(crop_ann_groups.keys()) | set(crop_offneg_groups.keys()))
n_keys = len(all_crop_keys)

for idx, crop_key in enumerate(all_crop_keys, 1):
    kt0 = time.perf_counter()
    dataset_id, well_id, crop_id = crop_key

    cand    = crop_union_groups.get(crop_key, pd.DataFrame()).copy()
    ann     = crop_ann_groups.get(crop_key, pd.DataFrame()).copy()
    off_neg = crop_offneg_groups.get(crop_key, pd.DataFrame()).copy()

    if len(cand) == 0 and len(ann) == 0 and len(off_neg) == 0:
        continue

    if len(cand) > 0:
        cand = cand.sort_values("union_id").reset_index(drop=True)

    if len(ann) == 0:
        ann = pd.DataFrame(columns=["annotation_id", "label", "truth_well_y_px", "truth_well_x_px"])
    else:
        ann = ann.copy()
        for _col, _dtype in [("annotation_id", "object"), ("label", "object"), ("truth_well_y_px", "float64"), ("truth_well_x_px", "float64")]:
            if _col not in ann.columns:
                ann[_col] = pd.Series(dtype=_dtype)
        ann = ann.sort_values("annotation_id").reset_index(drop=True)

    if len(off_neg) > 0:
        sort_cols = [c for c in ["raw_detection_id", "well_y_px", "well_x_px"] if c in off_neg.columns]
        off_neg = off_neg.sort_values(sort_cols).reset_index(drop=True)

    n_true = int((ann["label"] == "TRUE_SPOT").sum()) if "label" in ann.columns else 0
    n_neg  = int((ann["label"] == "EXPLICIT_NEGATIVE").sum()) if len(ann) else 0
    n_off  = int(len(off_neg))
    log(f"[{idx}/{n_keys}] {well_id}/{str(crop_id)[-12:]}  candidates={len(cand)}  true={n_true}  explicit_neg={n_neg}  offneg_side={n_off}")

    true_ann = ann.loc[ann["label"].eq("TRUE_SPOT")].reset_index(drop=True)
    neg_ann  = ann.loc[ann["label"].eq("EXPLICIT_NEGATIVE")].reset_index(drop=True)

    # Empty candidate slice can arrive as a 0-row, 0-col DataFrame from the by-key lookup.
    if cand is None or len(cand) == 0 or len(cand.columns) == 0:
        cand = pd.DataFrame(columns=["union_id", "union_medoid_well_y_px", "union_medoid_well_x_px"])

    # Candidate coordinate columns: prefer medoid, then centroid, then generic well coords.
    cand_coord_pairs = [
        ("union_medoid_well_y_px", "union_medoid_well_x_px"),
        ("union_centroid_well_y_px", "union_centroid_well_x_px"),
        ("well_y_px", "well_x_px"),
    ]
    cand_y_col = None
    cand_x_col = None
    for _y, _x in cand_coord_pairs:
        if _y in cand.columns and _x in cand.columns:
            cand_y_col, cand_x_col = _y, _x
            break

    # If there are still no coordinate cols, treat as empty-candidate crop rather than crash.
    if cand_y_col is None:
        cand_yx = np.empty((0, 2), dtype=np.float32)
    else:
        cand_yx = cand[[cand_y_col, cand_x_col]].to_numpy(dtype=np.float32)

    true_yx = (
        true_ann[["truth_well_y_px", "truth_well_x_px"]].to_numpy(dtype=np.float32)
        if len(true_ann) and {"truth_well_y_px", "truth_well_x_px"}.issubset(true_ann.columns)
        else np.empty((0, 2), dtype=np.float32)
    )

    neg_yx = (
        neg_ann[["truth_well_y_px", "truth_well_x_px"]].to_numpy(dtype=np.float32)
        if len(neg_ann) and {"truth_well_y_px", "truth_well_x_px"}.issubset(neg_ann.columns)
        else np.empty((0, 2), dtype=np.float32)
    )

    off_neg_yx = (
        off_neg[["well_y_px", "well_x_px"]].to_numpy(dtype=np.float32)
        if len(off_neg) and {"well_y_px", "well_x_px"}.issubset(off_neg.columns)
        else np.empty((0, 2), dtype=np.float32)
    )

    matched_cand_idx_orig = set()
    matched_true_idx      = set()
    hard_neg_cand_idx     = set()

    # All union candidates are eligible for true matching now; the off-channel negatives
    # live in a separate side table and no longer contaminate candidate_union itself.
    elig_indices = np.arange(len(cand), dtype=int)
    cand_yx_elig = cand_yx

    # ── Step 1: Hungarian on TRUE_SPOT ────────────────────────────────────
    kept, D_true = thresholded_hungarian(cand_yx_elig, true_yx, MATCH_RADIUS)

    for r, c, d in kept:
        orig_r = int(elig_indices[r])
        matched_cand_idx_orig.add(orig_r)
        matched_true_idx.add(int(c))

        crow = cand.iloc[orig_r]
        arow = true_ann.iloc[int(c)]

        MATCH_ROWS.append({
            "match_id":                  stable_match_id(crow["union_id"], arow["annotation_id"], "matched_positive", MATCH_RADIUS, MATCH_VERSION),
            "union_id":                  crow["union_id"],
            "annotation_id":             arow["annotation_id"],
            "dataset_id":                dataset_id,
            "well_id":                   well_id,
            "crop_id":                   crop_id,
            "match_status":              "matched_positive",
            "gt_label":                  "TRUE_SPOT",
            "gt_distance_px":            float(d),
            "gt_click_well_y_px":        float(arow["truth_well_y_px"]),
            "gt_click_well_x_px":        float(arow["truth_well_x_px"]),
            "candidate_well_y_px":       float(crow["union_medoid_well_y_px"]),
            "candidate_well_x_px":       float(crow["union_medoid_well_x_px"]),
            "annotation_confidence":     arow.get("confidence", np.nan),
            "annotation_label_original": arow.get("label", None),
            "truth_coord_source":        arow.get("truth_coord_source", None),
            "matching_algorithm":        CFG["matching_algorithm"],
            "truth_match_radius_px":     MATCH_RADIUS,
            "matching_registry_version": MATCH_VERSION,
        })

    # ── Step 2A: EXPLICIT_NEGATIVE annotations → matched_hard_negative ───
    if len(neg_ann) > 0 and len(cand) > 0:
        neg_tree = cKDTree(neg_yx)
        for ci in range(len(cand)):
            if ci in matched_cand_idx_orig or ci in hard_neg_cand_idx:
                continue
            dists, idxs = neg_tree.query(cand_yx[ci], k=1)
            if float(dists) <= EXPLICIT_NEG_RADIUS:
                hard_neg_cand_idx.add(ci)
                arow = neg_ann.iloc[int(idxs)]
                crow = cand.iloc[ci]
                MATCH_ROWS.append({
                    "match_id":                  stable_match_id(crow["union_id"], arow["annotation_id"], "matched_hard_negative", EXPLICIT_NEG_RADIUS, MATCH_VERSION),
                    "union_id":                  crow["union_id"],
                    "annotation_id":             arow["annotation_id"],
                    "dataset_id":                dataset_id,
                    "well_id":                   well_id,
                    "crop_id":                   crop_id,
                    "match_status":              "matched_hard_negative",
                    "gt_label":                  "EXPLICIT_NEGATIVE",
                    "gt_distance_px":            float(dists),
                    "gt_click_well_y_px":        float(arow["truth_well_y_px"]),
                    "gt_click_well_x_px":        float(arow["truth_well_x_px"]),
                    "candidate_well_y_px":       float(crow["union_medoid_well_y_px"]),
                    "candidate_well_x_px":       float(crow["union_medoid_well_x_px"]),
                    "annotation_confidence":     arow.get("confidence", np.nan),
                    "annotation_label_original": "EXPLICIT_NEGATIVE",
                    "truth_coord_source":        arow.get("truth_coord_source", None),
                    "matching_algorithm":        CFG["matching_algorithm"],
                    "truth_match_radius_px":     MATCH_RADIUS,
                    "matching_registry_version": MATCH_VERSION,
                })

    # ── Step 2B: side-table OFF_CHANNEL_NEGATIVE → matched_hard_negative ─
    if len(off_neg) > 0 and len(cand) > 0:
        off_tree = cKDTree(off_neg_yx)
        for ci in range(len(cand)):
            if ci in matched_cand_idx_orig or ci in hard_neg_cand_idx:
                continue
            dists, idxs = off_tree.query(cand_yx[ci], k=1)
            if float(dists) <= EXPLICIT_NEG_RADIUS:
                hard_neg_cand_idx.add(ci)
                arow = off_neg.iloc[int(idxs)]
                ann_id = arow["raw_detection_id"] if "raw_detection_id" in arow.index else f"offneg_{idxs}"
                MATCH_ROWS.append({
                    "match_id":                  stable_match_id(cand.iloc[ci]["union_id"], ann_id, "matched_hard_negative", EXPLICIT_NEG_RADIUS, MATCH_VERSION),
                    "union_id":                  cand.iloc[ci]["union_id"],
                    "annotation_id":             ann_id,
                    "dataset_id":                dataset_id,
                    "well_id":                   well_id,
                    "crop_id":                   crop_id,
                    "match_status":              "matched_hard_negative",
                    "gt_label":                  "OFF_CHANNEL_NEGATIVE",
                    "gt_distance_px":            float(dists),
                    "gt_click_well_y_px":        float(arow["well_y_px"]),
                    "gt_click_well_x_px":        float(arow["well_x_px"]),
                    "candidate_well_y_px":       float(cand.iloc[ci]["union_medoid_well_y_px"]),
                    "candidate_well_x_px":       float(cand.iloc[ci]["union_medoid_well_x_px"]),
                    "annotation_confidence":     np.nan,
                    "annotation_label_original": "OFF_CHANNEL_NEGATIVE",
                    "truth_coord_source":        "nb03_off_channel_negative_side_table",
                    "matching_algorithm":        CFG["matching_algorithm"],
                    "truth_match_radius_px":     MATCH_RADIUS,
                    "matching_registry_version": MATCH_VERSION,
                })

    # ── Step 3: unmatched candidates → negative (only when allowed) ──────
    is_neg_eligible = bool(cand.iloc[0].get("eligible_for_negative_supervision", False)) if len(cand) else False
    if (str(crop_id) in FULLY_ANNOTATED_CROPS) or is_neg_eligible:
        for ci in range(len(cand)):
            if ci in matched_cand_idx_orig or ci in hard_neg_cand_idx:
                continue
            crow = cand.iloc[ci]
            MATCH_ROWS.append({
                "match_id":                  stable_match_id(crow["union_id"], "NONE", "unmatched_candidate_negative", MATCH_RADIUS, MATCH_VERSION),
                "union_id":                  crow["union_id"],
                "annotation_id":             None,
                "dataset_id":                dataset_id,
                "well_id":                   well_id,
                "crop_id":                   crop_id,
                "match_status":              "unmatched_candidate_negative",
                "gt_label":                  "NEGATIVE",
                "gt_distance_px":            np.nan,
                "gt_click_well_y_px":        np.nan,
                "gt_click_well_x_px":        np.nan,
                "candidate_well_y_px":       float(cand_yx[ci][0]),
                "candidate_well_x_px":       float(cand_yx[ci][1]),
                "annotation_confidence":     np.nan,
                "annotation_label_original": None,
                "truth_coord_source":        None,
                "matching_algorithm":        CFG["matching_algorithm"],
                "truth_match_radius_px":     MATCH_RADIUS,
                "matching_registry_version": MATCH_VERSION,
            })

    # ── Step 4: unmatched truth → false negative (diagnostic only) ────────
    for ai in range(len(true_ann)):
        if ai in matched_true_idx:
            continue
        arow = true_ann.iloc[ai]
        nearest_cand_dist = (
            float(np.min(D_true[:, ai])) if D_true.size > 0 and D_true.shape[1] > ai
            else np.nan
        )
        MATCH_ROWS.append({
            "match_id":                  stable_match_id("NONE", arow["annotation_id"], "unmatched_truth_false_negative", MATCH_RADIUS, MATCH_VERSION),
            "union_id":                  None,
            "annotation_id":             arow["annotation_id"],
            "dataset_id":                dataset_id,
            "well_id":                   well_id,
            "crop_id":                   crop_id,
            "match_status":              "unmatched_truth_false_negative",
            "gt_label":                  "TRUE_SPOT",
            "gt_distance_px":            nearest_cand_dist,
            "gt_click_well_y_px":        float(arow["truth_well_y_px"]),
            "gt_click_well_x_px":        float(arow["truth_well_x_px"]),
            "candidate_well_y_px":       np.nan,
            "candidate_well_x_px":       np.nan,
            "annotation_confidence":     arow.get("confidence", np.nan),
            "annotation_label_original": "TRUE_SPOT",
            "truth_coord_source":        arow.get("truth_coord_source", None),
            "matching_algorithm":        CFG["matching_algorithm"],
            "truth_match_radius_px":     MATCH_RADIUS,
            "matching_registry_version": MATCH_VERSION,
        })
        TRUTH_SUMMARY_ROWS.append({
            "annotation_id":              arow["annotation_id"],
            "dataset_id":                 dataset_id,
            "well_id":                    well_id,
            "crop_id":                    crop_id,
            "label":                      "TRUE_SPOT",
            "truth_well_y_px":            float(arow["truth_well_y_px"]),
            "truth_well_x_px":            float(arow["truth_well_x_px"]),
            "match_status":               "unmatched_truth_false_negative",
            "matched_union_id":           None,
            "nearest_union_distance_px":  nearest_cand_dist,
            "matching_registry_version":  MATCH_VERSION,
        })

    n_matched = len(matched_cand_idx_orig)
    n_false_neg = len(true_ann) - len(matched_true_idx)
    n_hard_neg_crop = len(hard_neg_cand_idx)
    n_unmatched_neg = sum(
        1 for r in MATCH_ROWS
        if r["dataset_id"] == dataset_id
        and r["well_id"] == well_id
        and r["crop_id"] == crop_id
        and r["match_status"] == "unmatched_candidate_negative"
    )
    CROP_SUMMARY_ROWS.append({
        "dataset_id": dataset_id,
        "well_id": well_id,
        "crop_id": crop_id,
        "n_candidates": int(len(cand)),
        "n_true_annotations": int(len(true_ann)),
        "n_explicit_negative_annotations": int(len(neg_ann)),
        "n_off_channel_side_table": int(len(off_neg)),
        "n_matched": int(n_matched),
        "n_hard_negative": int(n_hard_neg_crop),
        "n_unmatched_negative": int(n_unmatched_neg),
        "n_false_negative": int(n_false_neg),
    })
    MATCH_TIMINGS.append({
        "dataset_id": dataset_id,
        "well_id": well_id,
        "crop_id": crop_id,
        "n_candidates": int(len(cand)),
        "n_annotations": int(len(ann)),
        "n_off_channel_side_table": int(len(off_neg)),
        "elapsed_sec": float(time.perf_counter() - kt0),
    })

candidate_to_truth_match = pd.DataFrame(MATCH_ROWS)
truth_evaluation_summary = pd.DataFrame(TRUTH_SUMMARY_ROWS)
matching_timing_df = pd.DataFrame(MATCH_TIMINGS)
crop_match_summary = pd.DataFrame(CROP_SUMMARY_ROWS)

log(f"Matching complete in {time.perf_counter() - t0_match:.2f}s")
log(f"candidate_to_truth_match: {len(candidate_to_truth_match):,} rows")
display(candidate_to_truth_match.head(10))


[2026-01-24 04:31:20 UTC] ==========================================================================================
[2026-01-24 04:31:20 UTC] BEGIN MATCHING
[2026-01-24 04:31:20 UTC] [1/50] C8/0adf818b2b6b  candidates=8  true=3  explicit_neg=0  offneg_side=19
[2026-01-24 04:31:20 UTC] [2/50] C8/1ffcd18f3eed  candidates=3  true=2  explicit_neg=0  offneg_side=11
[2026-01-24 04:31:20 UTC] [3/50] C8/2b359db36154  candidates=3  true=3  explicit_neg=0  offneg_side=9
[2026-01-24 04:31:20 UTC] [4/50] C8/2b6411e62d9d  candidates=106  true=1  explicit_neg=3  offneg_side=234
[2026-01-24 04:31:20 UTC] [5/50] C8/3f4741bac138  candidates=13  true=6  explicit_neg=0  offneg_side=34
[2026-01-24 04:31:20 UTC] [6/50] C8/495ce548be49  candidates=7  true=0  explicit_neg=1  offneg_side=17
[2026-01-24 04:31:20 UTC] [7/50] C8/49b3266fc66f  candidates=82  true=0  explicit_neg=5  offneg_side=186
[2026-01-24 04:31:20 UTC] [8/50] C8/4f61d31afbef  candidates=11  true=2  explicit_neg=0  offneg_side=26
[2026-01-24 

,match_id,union_id,annotation_id,dataset_id,well_id,crop_id,match_status,gt_label,gt_distance_px,gt_click_well_y_px,gt_click_well_x_px,candidate_well_y_px,candidate_well_x_px,annotation_confidence,annotation_label_original,truth_coord_source,matching_algorithm,truth_match_radius_px,matching_registry_version
0,match_583f9f5506fd79f0,586012895e02c21a247b24b5c231d0e2e3ab2b12,ann_907a221691c0c636,dataset_default,C8,crop_C8_0adf818b2b6b,matched_positive,TRUE_SPOT,1.000000,450.000000,4504.000000,451.000000,4504.000000,1.0,TRUE_SPOT,refined_well,hungarian_thresholded,8.0,matching_v1_crop_hungarian_medoid_refined_halo
1,match_07e40c41192a85bf,bed2dad68194464ad17c59214b298cffbb0a2e45,ann_9563cd172c3a864f,dataset_default,C8,crop_C8_0adf818b2b6b,matched_positive,TRUE_SPOT,1.414214,619.000000,4512.000000,618.000000,4513.000000,1.0,TRUE_SPOT,refined_well,hungarian_thresholded,8.0,matching_v1_crop_hungarian_medoid_refined_halo
2,match_137669018ef7eb94,ea486ce1ec17bb0744cd4e13fa92f905c62fd165,ann_fe0dc92ca60c7a40,dataset_default,C8,crop_C8_0adf818b2b6b,matched_positive,TRUE_SPOT,0.000000,464.000000,4689.000000,464.000000,4689.000000,1.0,TRUE_SPOT,refined_well,hungarian_thresholded,8.0,matching_v1_crop_hungarian_medoid_refined_halo
3,match_3bb23f1debcd923a,265743f6ea02f82a0a5acac18badf7585bea4432,6ffbc7f477eec1e63d5f6fa67bbde1af2a18e47a,dataset_default,C8,crop_C8_0adf818b2b6b,matched_hard_negative,OFF_CHANNEL_NEGATIVE,0.000000,456.000000,4514.000000,456.000000,4514.000000,NaN,OFF_CHANNEL_NEGATIVE,nb03_off_channel_negative_side_table,hungarian_thresholded,8.0,matching_v1_crop_hungarian_medoid_refined_halo
4,match_72ce75b93a694df5,6351df498e59b08ff56ffcd6a2b64615a09ad56f,e99b2f599a80bf3663a4e17770d4780ccc4bbffe,dataset_default,C8,crop_C8_0adf818b2b6b,matched_hard_negative,OFF_CHANNEL_NEGATIVE,0.122614,674.935059,4834.895996,675.000000,4835.000000,NaN,OFF_CHANNEL_NEGATIVE,nb03_off_channel_negative_side_table,hungarian_thresholded,8.0,matching_v1_crop_hungarian_medoid_refined_halo
5,match_882ddbc24df92a37,651ba7f1e5ccca3334c8fde8e3b22a15b505f659,27ba3c785298ad401ad584cd5d4a497d14318c25,dataset_default,C8,crop_C8_0adf818b2b6b,matched_hard_negative,OFF_CHANNEL_NEGATIVE,0.000000,728.366638,4830.200195,728.366638,4830.200195,NaN,OFF_CHANNEL_NEGATIVE,nb03_off_channel_negative_side_table,hungarian_thresholded,8.0,matching_v1_crop_hungarian_medoid_refined_halo
6,match_cc87379308a88479,78023a145d273708a8272cb22e732eee31df9c1d,a28ab64181f958d8110de9bd563cbee27e7e7fa3,dataset_default,C8,crop_C8_0adf818b2b6b,matched_hard_negative,OFF_CHANNEL_NEGATIVE,0.000000,385.500000,4489.000000,385.500000,4489.000000,NaN,OFF_CHANNEL_NEGATIVE,nb03_off_channel_negative_side_table,hungarian_thresholded,8.0,matching_v1_crop_hungarian_medoid_refined_halo
7,match_e04e053a8a204670,b0c833b1a3d971c374988aac2def49b6851e0d04,5673d649ee2345d668ff6a46029f969999b8fe26,dataset_default,C8,crop_C8_0adf818b2b6b,matched_hard_negative,OFF_CHANNEL_NEGATIVE,0.000000,720.000000,4838.000000,720.000000,4838.000000,NaN,OFF_CHANNEL_NEGATIVE,nb03_off_channel_negative_side_table,hungarian_thresholded,8.0,matching_v1_crop_hungarian_medoid_refined_halo
8,match_57d1e747f16fb36f,5d0757372ff87f3db4afca5fb3b894605cb9bf62,ann_78ad52ae1a9e7f63,dataset_default,C8,crop_C8_1ffcd18f3eed,matched_positive,TRUE_SPOT,2.000000,7366.000000,7190.000000,7366.000000,7188.000000,1.0,TRUE_SPOT,refined_well,hungarian_thresholded,8.0,matching_v1_crop_hungarian_medoid_refined_halo
9,match_8c81c3ecfa174eeb,db8d248667d9f7b957ddd255a4821722dcea5e2c,ann_06f7ca83424571c2,dataset_default,C8,crop_C8_1ffcd18f3eed,matched_positive,TRUE_SPOT,1.000000,7298.000000,7175.000000,7298.000000,7176.000000,1.0,TRUE_SPOT,refined_well,hungarian_thresholded,8.0,matching_v1_crop_hungarian_medoid_refined_halo


In [29]:
# -------------------------------------------------------------------
# MATCH TABLE VALIDATION  (§3.7)
# -------------------------------------------------------------------
required_match_cols = {
    "match_id", "union_id", "annotation_id", "match_status", "gt_label",
    "gt_distance_px", "matching_algorithm", "truth_match_radius_px",
    "matching_registry_version",
}
missing_match_cols = sorted(required_match_cols - set(candidate_to_truth_match.columns))
assert not missing_match_cols, f"candidate_to_truth_match missing columns: {missing_match_cols}"

allowed_match_status = {
    "matched_positive",
    "matched_hard_negative",
    "unmatched_candidate_negative",
    "unmatched_truth_false_negative",
}
bad_match_status = sorted(set(candidate_to_truth_match["match_status"]) - allowed_match_status)
assert not bad_match_status, f"Invalid match_status values: {bad_match_status}"
assert candidate_to_truth_match["match_id"].is_unique, "match_id must be unique"

# One-to-one for positives
_pos = candidate_to_truth_match[candidate_to_truth_match["match_status"] == "matched_positive"]
assert _pos["union_id"].is_unique,      "union_id appears in multiple matched_positive rows"
assert _pos["annotation_id"].is_unique, "annotation_id appears in multiple matched_positive rows"

# Matched positives must have both IDs
bad_pos = _pos[_pos["union_id"].isna() | _pos["annotation_id"].isna()]
assert len(bad_pos) == 0, "matched_positive rows must have both union_id and annotation_id"

n_pos  = int((candidate_to_truth_match["match_status"] == "matched_positive").sum())
n_neg  = int((candidate_to_truth_match["match_status"] == "unmatched_candidate_negative").sum())
n_fn   = int((candidate_to_truth_match["match_status"] == "unmatched_truth_false_negative").sum())
n_hard = int((candidate_to_truth_match["match_status"] == "matched_hard_negative").sum())

log(f"match_status counts: pos={n_pos}  neg={n_neg}  fn={n_fn}  hard_neg={n_hard}")
if n_pos + n_fn > 0:
    log(f"Detection recall (pre-classifier): {n_pos / (n_pos + n_fn):.3f}  ({n_pos}/{n_pos + n_fn})")
if n_pos + n_neg > 0:
    log(f"Positive rate in supervision:      {n_pos / (n_pos + n_neg):.4f}  ({n_pos}/{n_pos + n_neg})")
log("candidate_to_truth_match validation passed.")

[2026-01-24 04:31:45 UTC] match_status counts: pos=300  neg=0  fn=3  hard_neg=564
[2026-01-24 04:31:45 UTC] Detection recall (pre-classifier): 0.990  (300/303)
[2026-01-24 04:31:45 UTC] Positive rate in supervision:      1.0000  (300/300)
[2026-01-24 04:31:45 UTC] candidate_to_truth_match validation passed.


In [30]:
# -------------------------------------------------------------------
# MATCH DIAGNOSTICS
# -------------------------------------------------------------------
log("=" * 90)
log("MATCH DIAGNOSTICS")

matched   = candidate_to_truth_match[candidate_to_truth_match["match_status"] == "matched_positive"]
false_neg = candidate_to_truth_match[candidate_to_truth_match["match_status"] == "unmatched_truth_false_negative"]

if len(matched) > 0:
    dists = matched["gt_distance_px"].dropna()
    log(f"\nMatched pair distances (px): mean={dists.mean():.2f}  median={dists.median():.2f}  "
        f"std={dists.std():.2f}  p90={dists.quantile(0.9):.2f}  max={dists.max():.2f}")

if len(false_neg) > 0:
    fn_dists = false_neg["gt_distance_px"].dropna()
    log(f"\nFalse negatives ({len(false_neg)}): nearest cand dist: "
        f"mean={fn_dists.mean():.2f}  median={fn_dists.median():.2f}  "
        f"min={fn_dists.min():.2f}  max={fn_dists.max():.2f}")
    n_near = int((fn_dists < MATCH_RADIUS * 2).sum())
    log(f"  Within 2× radius ({MATCH_RADIUS*2:.0f}px): {n_near} near-misses | {len(fn_dists) - n_near} detection failures")

log(f"\nRecall sensitivity to matching radius:")
total_truth = n_pos + n_fn
for test_r in [3.0, 5.0, 8.0, 10.0, 12.0, 15.0]:
    n_at_r = int((matched["gt_distance_px"] <= test_r).sum()) if len(matched) > 0 else 0
    log(f"  r={test_r:.0f}px  recall={n_at_r / max(total_truth, 1):.3f}  ({n_at_r}/{total_truth})")

# Per-crop summary table
match_counts_pivot = (
    candidate_to_truth_match
    .groupby(["dataset_id", "well_id", "crop_id", "match_status"], dropna=False)
    .size().rename("n_rows").reset_index()
    .pivot_table(index=["dataset_id", "well_id", "crop_id"], columns="match_status", values="n_rows", aggfunc="sum", fill_value=0)
    .reset_index()
)
for col in ["matched_positive", "unmatched_candidate_negative", "unmatched_truth_false_negative", "excluded_uncertain"]:
    if col not in match_counts_pivot.columns:
        match_counts_pivot[col] = 0
match_counts_pivot["precision_proxy"] = (
    match_counts_pivot["matched_positive"] /
    (match_counts_pivot["matched_positive"] + match_counts_pivot["unmatched_candidate_negative"] + 1e-9)
)
match_counts_pivot["recall_proxy"] = (
    match_counts_pivot["matched_positive"] /
    (match_counts_pivot["matched_positive"] + match_counts_pivot["unmatched_truth_false_negative"] + 1e-9)
)
display(match_counts_pivot.sort_values(["well_id", "crop_id"]))

# Diagnostic plots
if CFG["write_match_diagnostics"] and len(matched) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].hist(matched["gt_distance_px"].dropna(), bins=30, edgecolor="black", linewidth=0.5)
    axes[0].axvline(MATCH_RADIUS, color="red", linestyle="--", label=f"radius={MATCH_RADIUS}px")
    axes[0].set_xlabel("match distance (px)"); axes[0].set_ylabel("count")
    axes[0].set_title("Matched pair distances"); axes[0].legend()

    if len(crop_match_summary) > 0:
        x = range(len(crop_match_summary))
        axes[1].bar(x, crop_match_summary["n_matched"], label="matched", alpha=0.8)
        axes[1].bar(x, crop_match_summary["n_false_negative"],
                    bottom=crop_match_summary["n_matched"], label="false neg", alpha=0.8)
        axes[1].set_xlabel("crop index"); axes[1].set_ylabel("count")
        axes[1].set_title("Per-crop: matched + FN"); axes[1].legend()

    if len(false_neg) > 0:
        axes[2].hist(false_neg["gt_distance_px"].dropna(), bins=30, edgecolor="black", linewidth=0.5)
        axes[2].axvline(MATCH_RADIUS, color="red", linestyle="--", label=f"radius={MATCH_RADIUS}px")
        axes[2].set_xlabel("nearest candidate distance (px)")
        axes[2].set_title("FN: dist to nearest candidate"); axes[2].legend()

    fig.tight_layout()
    diag_path = REPORT_DIR / f"{RUN_ID}_match_diagnostics.png"
    fig.savefig(diag_path, dpi=CFG["qa_fig_dpi"], bbox_inches="tight")
    plt.close(fig)
    log(f"Diagnostics plot saved: {diag_path.name}")

[2026-01-24 04:31:57 UTC] ==========================================================================================
[2026-01-24 04:31:57 UTC] MATCH DIAGNOSTICS
[2026-01-24 04:31:57 UTC] 
Matched pair distances (px): mean=1.63  median=1.00  std=1.68  p90=4.12  max=7.28
[2026-01-24 04:31:57 UTC] 
False negatives (3): nearest cand dist: mean=26.29  median=10.00  min=6.08  max=62.80
[2026-01-24 04:31:57 UTC]   Within 2× radius (16px): 2 near-misses | 1 detection failures
[2026-01-24 04:31:57 UTC] 
Recall sensitivity to matching radius:
[2026-01-24 04:31:57 UTC]   r=3px  recall=0.805  (244/303)
[2026-01-24 04:31:57 UTC]   r=5px  recall=0.944  (286/303)
[2026-01-24 04:31:57 UTC]   r=8px  recall=0.990  (300/303)
[2026-01-24 04:31:57 UTC]   r=10px  recall=0.990  (300/303)
[2026-01-24 04:31:57 UTC]   r=12px  recall=0.990  (300/303)
[2026-01-24 04:31:57 UTC]   r=15px  recall=0.990  (300/303)


match_status,dataset_id,well_id,crop_id,matched_hard_negative,matched_positive,unmatched_truth_false_negative,unmatched_candidate_negative,excluded_uncertain,precision_proxy,recall_proxy
0,dataset_default,C8,crop_C8_0adf818b2b6b,5,3,0,0,0,1.0,1.000000
1,dataset_default,C8,crop_C8_1ffcd18f3eed,1,2,0,0,0,1.0,1.000000
2,dataset_default,C8,crop_C8_2b359db36154,0,3,0,0,0,1.0,1.000000
3,dataset_default,C8,crop_C8_2b6411e62d9d,105,1,0,0,0,1.0,1.000000
4,dataset_default,C8,crop_C8_3f4741bac138,7,6,0,0,0,1.0,1.000000
5,dataset_default,C8,crop_C8_495ce548be49,7,0,0,0,0,0.0,0.000000
6,dataset_default,C8,crop_C8_49b3266fc66f,82,0,0,0,0,0.0,0.000000
7,dataset_default,C8,crop_C8_4f61d31afbef,9,2,0,0,0,1.0,1.000000
8,dataset_default,C8,crop_C8_67faf4382509,2,1,0,0,0,1.0,1.000000
9,dataset_default,C8,crop_C8_70420ab6a23a,3,4,0,0,0,1.0,1.000000


[2026-01-24 04:31:58 UTC] Diagnostics plot saved: nb05_20260124T042657Z_2af2592e_match_diagnostics.png


In [31]:
# -------------------------------------------------------------------
# SPATIAL BLOCK SPLITS  (§3.10, §10.2)
#
# Each annotated QA crop window = one spatial block over the masked full-well candidate universe.
# Deterministic hash-based assignment, stratified by well_id.
# Always regenerates splits from the current annotated crop set.
# -------------------------------------------------------------------
log("=" * 90)
log("SPATIAL BLOCK SPLITS")

SPLIT_REGISTRY_VERSION = CFG["split_registry_version"]
existing_splits_path   = None
splits_df              = None

if splits_df is None:
    TRAIN_FRAC = CFG["split_train_frac"]
    CAL_FRAC   = CFG["split_cal_frac"]
    SPLIT_SEED = CFG["split_seed"]

    def deterministic_split_role(crop_id: str, well_id: str, seed: int) -> str:
        h   = hashlib.sha256(f"{crop_id}|{well_id}|{seed}".encode()).hexdigest()
        val = int(h[:8], 16) / 0xFFFFFFFF
        if val < TRAIN_FRAC:
            return "train"
        elif val < TRAIN_FRAC + CAL_FRAC:
            return "calibration"
        return "test"

    crop_reg_idx = crop_registry.set_index("crop_id")
    split_rows = []
    for cid in sorted(FULLY_ANNOTATED_CROPS):
        if cid not in crop_reg_idx.index:
            log(f"[warn] crop_id {cid} not found in crop_registry — skipping split assignment")
            continue
        cr   = crop_reg_idx.loc[cid]
        role = deterministic_split_role(cid, cr["well_id"], SPLIT_SEED)
        split_rows.append({
            "split_id":              sha1_text(f"{cid}|{role}|{SPLIT_REGISTRY_VERSION}"),
            "dataset_id":            cr["dataset_id"],
            "well_id":               cr["well_id"],
            "crop_id":               cid,
            "spatial_block_id":      cid,
            "split_role":            role,
            "split_registry_version": SPLIT_REGISTRY_VERSION,
        })
    splits_df = pd.DataFrame(split_rows)
    log(f"Generated {len(splits_df)} split assignments.")

assert splits_df["crop_id"].is_unique, "Split leakage: crop_id appears in multiple splits!"
assert set(splits_df["split_role"]).issubset({"train", "calibration", "test"})

role_counts = splits_df["split_role"].value_counts()
log(f"\nSplit assignment ({len(splits_df)} crops):")
for role in ["train", "calibration", "test"]:
    log(f"  {role}: {role_counts.get(role, 0)} crops")
log("\nPer-well split distribution:")
display(splits_df.groupby(["well_id", "split_role"]).size().unstack(fill_value=0))
log("No split leakage ✓")

[2026-01-24 04:32:21 UTC] ==========================================================================================
[2026-01-24 04:32:21 UTC] SPATIAL BLOCK SPLITS
[2026-01-24 04:32:21 UTC] Generated 50 split assignments.
[2026-01-24 04:32:21 UTC] 
Split assignment (50 crops):
[2026-01-24 04:32:21 UTC]   train: 37 crops
[2026-01-24 04:32:21 UTC]   calibration: 7 crops
[2026-01-24 04:32:21 UTC]   test: 6 crops
[2026-01-24 04:32:21 UTC] 
Per-well split distribution:


split_role,calibration,test,train
well_id,,,
C8,5,3,17
D8,2,3,20


[2026-01-24 04:32:21 UTC] No split leakage ✓


In [32]:
# -------------------------------------------------------------------
# BUILD training_supervision_table  (§3.9)
#
# Left-join masked full-well mega_feature_table + match results + splits.
# Every union_id gets exactly one row.
# supervision_status ∈ {included, excluded}
# target_binary ∈ {0, 1, NaN}  (NaN for excluded rows)
# -------------------------------------------------------------------
log("=" * 90)
log("BUILDING training_supervision_table")

# ── Supervision mapping ────────────────────────────────────────────────────
def supervision_mapping(row) -> tuple:
    ms = row["match_status"]
    gl = str(row.get("gt_label", ""))
    if ms == "matched_positive":
        return "included", 1.0, "matched_true_spot"
    elif ms == "matched_hard_negative":
        if gl == "EXPLICIT_NEGATIVE":
            return "included", 0.0, "matched_explicit_negative_annotation"
        elif gl == "OFF_CHANNEL_NEGATIVE":
            return "included", 0.0, "matched_off_channel_negative_side_table"
        else:
            return "included", 0.0, "matched_hard_negative"
    elif ms == "unmatched_candidate_negative":
        return "included", 0.0, "annotated_territory_unmatched_candidate"
    else:
        return "excluded", np.nan, f"unsupported_{ms}"

union_backed = candidate_to_truth_match[candidate_to_truth_match["union_id"].notna()].copy()
mapped       = union_backed.apply(supervision_mapping, axis=1, result_type="expand")
mapped.columns = ["supervision_status", "target_binary", "target_source"]
union_backed = pd.concat([union_backed.reset_index(drop=True), mapped], axis=1)

# ── Sample weight computation ──────────────────────────────────────────────
supervised_only = union_backed[union_backed["target_binary"].notna()]
n_pos_w = int(supervised_only["target_binary"].sum())
n_neg_w = int(len(supervised_only) - n_pos_w)
n_total_w = n_pos_w + n_neg_w

if CFG["positive_sample_weight_override"] is not None:
    w_pos = float(CFG["positive_sample_weight_override"])
    w_neg = float(CFG["negative_sample_weight_override"] or 1.0)
    log(f"Sample weights (override): w_pos={w_pos:.4f}, w_neg={w_neg:.4f}")
else:
    strategy = CFG["sample_weight_strategy"]
    if strategy == "balanced":
        w_pos = n_total_w / (2 * max(n_pos_w, 1))
        w_neg = n_total_w / (2 * max(n_neg_w, 1))
    elif strategy == "sqrt_inverse_freq":
        freq_pos = n_pos_w / max(n_total_w, 1)
        freq_neg = n_neg_w / max(n_total_w, 1)
        w_pos = 1.0 / max(math.sqrt(freq_pos), 1e-6)
        w_neg = 1.0 / max(math.sqrt(freq_neg), 1e-6)
        w_mean = (w_pos * n_pos_w + w_neg * n_neg_w) / max(n_total_w, 1)
        w_pos /= max(w_mean, 1e-6)
        w_neg /= max(w_mean, 1e-6)
    else:  # "none"
        w_pos = 1.0
        w_neg = 1.0
    log(f"Sample weights ({strategy}): w_pos={w_pos:.4f}, w_neg={w_neg:.4f}")

# Base weights
union_backed["sample_weight"] = (
    union_backed["target_binary"]
    .map({1.0: w_pos, 0.0: w_neg})
    .fillna(float(CFG["excluded_sample_weight"]))
)
# Hard negatives get higher weight — these are explicitly annotated "not a spot"
hard_neg_mask = union_backed["target_source"] == "matched_explicit_negative"
hard_neg_multiplier = CFG.get("hard_negative_weight_multiplier", 3.0)
union_backed.loc[hard_neg_mask, "sample_weight"] *= hard_neg_multiplier
n_hard_neg = int(hard_neg_mask.sum())
if n_hard_neg > 0:
    log(f"Hard negative weight: {n_hard_neg} rows × {hard_neg_multiplier}x multiplier")

# ── Join splits ────────────────────────────────────────────────────────────
union_backed = union_backed.merge(
    splits_df[["crop_id", "split_id", "split_role"]],
    on="crop_id", how="left"
)

# ── Left-join mega_feature_table (primary) + supervision results ───────────
# Exclude heavy provenance cols from mega to keep the supervision table clean
_meta_exclude = {"run_id", "nb04_run_id", "project_config_version",
                 "feature_registry_version", "created_at_utc"}
mega_cols_to_join = [c for c in mega_feature_table.columns if c not in _meta_exclude]

training_supervision_table = mega_feature_table[mega_cols_to_join].merge(
    union_backed[[
        "union_id", "split_id", "split_role", "supervision_status",
        "target_binary", "target_source", "sample_weight",
        "match_status", "annotation_id", "gt_distance_px", "gt_label",
    ]],
    on="union_id",
    how="left",
)

# ── Join OFF-aware augmentation table from appended NB03/04 caches ─────────
OFFAUG_JOINED_COLS = []
if CFG.get("join_offaware_aug_into_training_supervision", True) and "f459_offaware_aug" in globals() and len(f459_offaware_aug):
    _aug = f459_offaware_aug.copy()
    _aug["union_id"] = _aug["union_id"].astype(str)
    assert _aug["union_id"].is_unique, "f459_offaware_aug must have unique union_id before join"
    aug_cols = [c for c in _aug.columns if c != "union_id"]
    dup_cols = [c for c in aug_cols if c in training_supervision_table.columns]
    if dup_cols:
        _aug = _aug.rename(columns={c: f"{c}__offaug" for c in dup_cols})
        aug_cols = [c for c in _aug.columns if c != "union_id"]
    training_supervision_table["union_id"] = training_supervision_table["union_id"].astype(str)
    training_supervision_table = training_supervision_table.merge(_aug, on="union_id", how="left")
    OFFAUG_JOINED_COLS = aug_cols
    log(f"Joined OFF-aware augmentation columns into training_supervision_table: n_cols={len(aug_cols)}")
else:
    log("No OFF-aware augmentation table joined into training_supervision_table")

# Fill unsupervised candidates (outside annotated territory)
training_supervision_table["supervision_status"] = training_supervision_table["supervision_status"].fillna("excluded")
training_supervision_table["target_source"]      = training_supervision_table["target_source"].fillna("outside_supervision_territory")
training_supervision_table["sample_weight"]      = training_supervision_table["sample_weight"].fillna(0.0)
training_supervision_table["target_binary"]      = pd.to_numeric(training_supervision_table["target_binary"], errors="coerce")

log(f"training_supervision_table: {len(training_supervision_table):,} rows × {len(training_supervision_table.columns)} cols")
log(f"Class balance: {n_pos_w:,} positive / {n_neg_w:,} negative  (total supervised {n_total_w:,})")
log(f"Positive rate: {n_pos_w / max(n_total_w, 1):.4f}")

log("\nPer-split class distribution:")
_sup = training_supervision_table[training_supervision_table["target_binary"].notna()]
_split_summary = (
    _sup.groupby("split_role")["target_binary"]
    .agg([("n_positive", "sum"), ("n_total", "count"), ("pos_rate", "mean")])
)
_split_summary["n_negative"] = _split_summary["n_total"] - _split_summary["n_positive"]
display(_split_summary[["n_positive", "n_negative", "n_total", "pos_rate"]])

[2026-01-24 04:32:22 UTC] ==========================================================================================
[2026-01-24 04:32:22 UTC] BUILDING training_supervision_table
[2026-01-24 04:32:22 UTC] Sample weights (balanced): w_pos=1.4400, w_neg=0.7660
[2026-01-24 04:32:22 UTC] No OFF-aware augmentation table joined into training_supervision_table
[2026-01-24 04:32:22 UTC] training_supervision_table: 864 rows × 450 cols
[2026-01-24 04:32:22 UTC] Class balance: 300 positive / 564 negative  (total supervised 864)
[2026-01-24 04:32:22 UTC] Positive rate: 0.3472
[2026-01-24 04:32:22 UTC] 
Per-split class distribution:


,n_positive,n_negative,n_total,pos_rate
split_role,,,,
calibration,16.0,155.0,171,0.093567
test,31.0,35.0,66,0.469697
train,252.0,372.0,624,0.403846


In [33]:
# -------------------------------------------------------------------
# SUPERVISION TABLE VALIDATION  (§3.9)
# -------------------------------------------------------------------
required_supervision_cols = {
    "union_id", "split_id", "supervision_status",
    "target_binary", "target_source", "sample_weight",
}
missing_sup = sorted(required_supervision_cols - set(training_supervision_table.columns))
assert not missing_sup, f"training_supervision_table missing columns: {missing_sup}"
assert training_supervision_table["union_id"].is_unique, "training_supervision_table: must be one row per union_id"

allowed_sup_status = {"included", "excluded"}
bad_sup = sorted(set(training_supervision_table["supervision_status"]) - allowed_sup_status)
assert not bad_sup, f"Invalid supervision_status values: {bad_sup}"

included = training_supervision_table["supervision_status"] == "included"
bad_targets = training_supervision_table[included & ~training_supervision_table["target_binary"].isin([0.0, 1.0])]
assert len(bad_targets) == 0, f"{len(bad_targets)} included rows have non-binary target"

excluded_nonzero = training_supervision_table[
    (training_supervision_table["supervision_status"] == "excluded") &
    (training_supervision_table["sample_weight"] != 0.0)
]
assert len(excluded_nonzero) == 0, "Excluded rows must have sample_weight == 0.0"

# Row count must match mega_feature_table
assert len(training_supervision_table) == len(mega_feature_table), (
    f"Row count mismatch: supervision_table={len(training_supervision_table)} vs mega={len(mega_feature_table)}"
)

log("training_supervision_table validation passed.")
display(training_supervision_table[["union_id", "crop_id", "supervision_status", "target_binary", "target_source", "split_role", "sample_weight"]].head(10))

[2026-01-24 04:32:23 UTC] training_supervision_table validation passed.


,union_id,crop_id,supervision_status,target_binary,target_source,split_role,sample_weight
0,586012895e02c21a247b24b5c231d0e2e3ab2b12,crop_C8_0adf818b2b6b,included,1.0,matched_true_spot,train,1.440000
1,bed2dad68194464ad17c59214b298cffbb0a2e45,crop_C8_0adf818b2b6b,included,1.0,matched_true_spot,train,1.440000
2,6351df498e59b08ff56ffcd6a2b64615a09ad56f,crop_C8_0adf818b2b6b,included,0.0,matched_off_channel_negative_side_table,train,0.765957
3,ea486ce1ec17bb0744cd4e13fa92f905c62fd165,crop_C8_0adf818b2b6b,included,1.0,matched_true_spot,train,1.440000
4,78023a145d273708a8272cb22e732eee31df9c1d,crop_C8_0adf818b2b6b,included,0.0,matched_off_channel_negative_side_table,train,0.765957
5,b0c833b1a3d971c374988aac2def49b6851e0d04,crop_C8_0adf818b2b6b,included,0.0,matched_off_channel_negative_side_table,train,0.765957
6,651ba7f1e5ccca3334c8fde8e3b22a15b505f659,crop_C8_0adf818b2b6b,included,0.0,matched_off_channel_negative_side_table,train,0.765957
7,265743f6ea02f82a0a5acac18badf7585bea4432,crop_C8_0adf818b2b6b,included,0.0,matched_off_channel_negative_side_table,train,0.765957
8,5d0757372ff87f3db4afca5fb3b894605cb9bf62,crop_C8_1ffcd18f3eed,included,1.0,matched_true_spot,train,1.440000
9,44a64c68f45847bc5b6f6a9f38016a5a88c77532,crop_C8_1ffcd18f3eed,included,0.0,matched_off_channel_negative_side_table,train,0.765957


In [34]:
# -------------------------------------------------------------------
# QA SUMMARIES
# -------------------------------------------------------------------
log("=" * 90)
log("QA SUMMARIES")

# Per-well class distribution
log("\nPer-well class distribution:")
_sup = training_supervision_table[training_supervision_table["target_binary"].notna()]
display(
    _sup.groupby("well_id")["target_binary"]
    .agg(n_pos="sum", n_total="count", pos_rate="mean")
)

# Per-crop class distribution + warnings
log("\nPer-crop class distribution:")
crop_class = _sup.groupby("crop_id")["target_binary"].agg(n_pos="sum", n_total="count", pos_rate="mean")
display(crop_class)
for cid, row in crop_class.iterrows():
    if row["n_pos"] == 0:
        log(f"  [warn] crop {cid}: 0 positives (all negative)")
    if row["pos_rate"] > 0.5:
        log(f"  [warn] crop {cid}: pos_rate={row['pos_rate']:.2f} > 50% — unusual")

# Split leakage check
crop_splits = training_supervision_table.dropna(subset=["split_role"]).groupby("crop_id")["split_role"].nunique()
leaky = crop_splits[crop_splits > 1]
assert len(leaky) == 0, f"Split leakage: {len(leaky)} crops appear in multiple splits: {leaky.index.tolist()}"
log("No split leakage ✓")

# Feature separability: Cohen's d between positive and negative classes
log("\nTop features by class separation (Cohen's d):")
_meta_cols = {
    "union_id", "split_id", "split_role", "supervision_status", "target_binary",
    "target_source", "sample_weight", "match_status", "annotation_id",
    "gt_distance_px", "gt_label", "dataset_id", "well_id", "crop_id", "cluster_id",
    "well_y_px", "well_x_px", "union_centroid_well_y_px", "union_centroid_well_x_px",
    "in_annotated_crop", "annotation_coverage_status", "is_completed_for_negatives",
    "eligible_for_negative_supervision", "coverage_allows_negative",
}
feature_cols_in_sup = [
    c for c in training_supervision_table.columns
    if c not in _meta_cols
    and pd.api.types.is_numeric_dtype(training_supervision_table[c])
]
separability = []
for col in feature_cols_in_sup:
    pos_v = _sup.loc[_sup["target_binary"] == 1, col].dropna()
    neg_v = _sup.loc[_sup["target_binary"] == 0, col].dropna()
    if len(pos_v) > 1 and len(neg_v) > 1:
        pooled_std = math.sqrt(
            (pos_v.var() * len(pos_v) + neg_v.var() * len(neg_v))
            / max(len(pos_v) + len(neg_v) - 2, 1)
        )
        d = (float(pos_v.mean()) - float(neg_v.mean())) / max(pooled_std, 1e-8)
        separability.append({"feature": col, "cohens_d": d, "abs_d": abs(d)})
if separability:
    sep_df = pd.DataFrame(separability).sort_values("abs_d", ascending=False)
    display(sep_df.head(20))

[2026-01-24 04:32:24 UTC] ==========================================================================================
[2026-01-24 04:32:24 UTC] QA SUMMARIES
[2026-01-24 04:32:24 UTC] 
Per-well class distribution:


,n_pos,n_total,pos_rate
well_id,,,
C8,52.0,429,0.121212
D8,247.0,432,0.571759


[2026-01-24 04:32:24 UTC] 
Per-crop class distribution:


,n_pos,n_total,pos_rate
crop_id,,,
crop_C8_0adf818b2b6b,3.0,8,0.375000
crop_C8_1ffcd18f3eed,2.0,3,0.666667
crop_C8_2b359db36154,3.0,3,1.000000
crop_C8_2b6411e62d9d,1.0,106,0.009434
crop_C8_3f4741bac138,6.0,13,0.461538
crop_C8_495ce548be49,0.0,7,0.000000
crop_C8_49b3266fc66f,0.0,82,0.000000
crop_C8_4f61d31afbef,2.0,11,0.181818
crop_C8_67faf4382509,1.0,3,0.333333


[2026-01-24 04:32:24 UTC]   [warn] crop crop_C8_1ffcd18f3eed: pos_rate=0.67 > 50% — unusual
[2026-01-24 04:32:24 UTC]   [warn] crop crop_C8_2b359db36154: pos_rate=1.00 > 50% — unusual
[2026-01-24 04:32:24 UTC]   [warn] crop crop_C8_495ce548be49: 0 positives (all negative)
[2026-01-24 04:32:24 UTC]   [warn] crop crop_C8_49b3266fc66f: 0 positives (all negative)
[2026-01-24 04:32:24 UTC]   [warn] crop crop_C8_70420ab6a23a: pos_rate=0.57 > 50% — unusual
[2026-01-24 04:32:24 UTC]   [warn] crop crop_C8_7c6b50d894ab: 0 positives (all negative)
[2026-01-24 04:32:24 UTC]   [warn] crop crop_C8_9bd27518736b: 0 positives (all negative)
[2026-01-24 04:32:24 UTC]   [warn] crop crop_C8_f7f7c4173888: 0 positives (all negative)
[2026-01-24 04:32:24 UTC]   [warn] crop crop_D8_0a071dcf71bd: pos_rate=0.94 > 50% — unusual
[2026-01-24 04:32:24 UTC]   [warn] crop crop_D8_2019d4c516b9: pos_rate=0.86 > 50% — unusual
[2026-01-24 04:32:24 UTC]   [warn] crop crop_D8_2194e1b20725: pos_rate=0.64 > 50% — unusual
[20

,feature,cohens_d,abs_d
201,proposer_support_count,2.674193,2.674193
202,proposer_support_fraction,2.674193,2.674193
265,method_diversity_frac,2.674193,2.674193
203,family_support_count,2.635443,2.635443
264,family_diversity_frac,2.635443,2.635443
224,vote_sf_G_union,2.582586,2.582586
204,union_n_members,2.452622,2.452622
213,vote_p04_dog_1_2,2.447324,2.447324
268,agree_starfish_and_dog,2.420377,2.420377
225,vote_fam_dog,2.401465,2.401465


In [35]:
# -------------------------------------------------------------------
# OPTIONAL QA OVERLAYS
# Renders per-crop overlays for worst-FN then worst-precision crops.
# Degrades gracefully if image paths are not resolvable from crop registry.
# -------------------------------------------------------------------
def try_find_crop_base_image(crop_row: pd.Series):
    candidate_cols = [c for c in crop_row.index if "path" in str(c).lower() or "file" in str(c).lower()]
    for c in candidate_cols:
        v = crop_row[c]
        if isinstance(v, str) and len(v.strip()):
            p = Path(v)
            if p.exists():
                return p
    return None

def read_image_2d(path) -> np.ndarray:
    arr = np.asarray(tifffile.imread(str(path)))
    arr = np.squeeze(arr)
    if arr.ndim == 3:
        arr = arr.max(axis=0)
    if arr.ndim != 2:
        arr = arr.reshape(arr.shape[-2], arr.shape[-1])
    return arr.astype(np.float32, copy=False)

def choose_overlay_crops(summary_df: pd.DataFrame, max_n: int) -> pd.DataFrame:
    if len(summary_df) == 0:
        return summary_df
    return (
        summary_df
        .sort_values(
            ["unmatched_truth_false_negative", "unmatched_candidate_negative", "matched_positive"],
            ascending=[False, False, False]
        )
        .head(max_n)
        .reset_index(drop=True)
    )

COLOR_MAP = {
    "matched_positive":               "lime",
    "unmatched_candidate_negative":   "red",
    "excluded_uncertain":             "gold",
    "unmatched_truth_false_negative": "cyan",
}

overlay_crops = choose_overlay_crops(match_counts_pivot, CFG["qa_max_overlay_crops"])
crop_status_idx = crop_status.set_index(["dataset_id", "well_id", "crop_id"])

if len(overlay_crops) == 0:
    log("No overlay crops to render.")
else:
    log(f"Rendering up to {len(overlay_crops)} QA overlay crops ...")
    for _, row in overlay_crops.iterrows():
        ckey = (row["dataset_id"], row["well_id"], row["crop_id"])
        if ckey not in crop_status_idx.index:
            continue
        crop_row = crop_status_idx.loc[ckey]
        img_path = try_find_crop_base_image(crop_row)
        if img_path is None:
            log(f"  [overlay skip] no resolvable image for crop={row['crop_id']}")
            continue
        try:
            full = read_image_2d(img_path)
        except Exception as exc:
            log(f"  [overlay skip] crop={row['crop_id']}: {type(exc).__name__}: {exc}")
            continue

        y0 = int(crop_row["well_ymin_px"]); x0 = int(crop_row["well_xmin_px"])
        y1 = int(crop_row["well_ymax_px"]); x1 = int(crop_row["well_xmax_px"])
        patch = full[y0:y1, x0:x1]
        lo, hi = np.percentile(patch, 1), np.percentile(patch, 99)
        patch_display = np.clip((patch - lo) / max(hi - lo, 1e-8), 0, 1)

        cand_sub = candidate_to_truth_match[
            (candidate_to_truth_match["dataset_id"] == row["dataset_id"]) &
            (candidate_to_truth_match["well_id"] == row["well_id"]) &
            (candidate_to_truth_match["crop_id"] == row["crop_id"])
        ]

        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(patch_display, cmap="gray", origin="upper")

        for status, g in cand_sub.groupby("match_status", dropna=False):
            col = COLOR_MAP.get(status, "white")
            g_cand = g[g["union_id"].notna() & np.isfinite(g["candidate_well_y_px"]) & np.isfinite(g["candidate_well_x_px"])]
            if len(g_cand):
                ax.scatter(
                    g_cand["candidate_well_x_px"] - x0,
                    g_cand["candidate_well_y_px"] - y0,
                    s=28, facecolors="none", edgecolors=col,
                    linewidths=1.3, alpha=CFG["qa_point_alpha"], label=f"{status} (cand)"
                )
            g_truth = g[g["annotation_id"].notna() & np.isfinite(g["gt_click_well_y_px"]) & np.isfinite(g["gt_click_well_x_px"])]
            if len(g_truth):
                ax.scatter(
                    g_truth["gt_click_well_x_px"] - x0,
                    g_truth["gt_click_well_y_px"] - y0,
                    s=40, marker="x", c=col,
                    linewidths=1.5, alpha=CFG["qa_point_alpha"], label=f"{status} (truth)"
                )

        ax.legend(loc="upper right", fontsize=7, framealpha=0.7)
        ax.set_title(
            f"{row['well_id']} | {row['crop_id'][-12:]}\n"
            f"pos={int(row.get('matched_positive',0))}  "
            f"neg={int(row.get('unmatched_candidate_negative',0))}  "
            f"fn={int(row.get('unmatched_truth_false_negative',0))}  "
            f"excl={int(row.get('excluded_uncertain',0))}",
            fontsize=9
        )
        ax.set_axis_off()
        fig.tight_layout()
        out_png = REPORT_DIR / f"{RUN_ID}_{row['crop_id']}_overlay.png"
        fig.savefig(out_png, dpi=CFG["qa_fig_dpi"], bbox_inches="tight")
        plt.show()
        plt.close(fig)
        log(f"  Saved: {out_png.name}")

[2026-01-24 04:32:24 UTC] Rendering up to 8 QA overlay crops ...
[2026-01-24 04:32:24 UTC]   [overlay skip] no resolvable image for crop=crop_D8_c59003e258cd
[2026-01-24 04:32:24 UTC]   [overlay skip] no resolvable image for crop=crop_D8_581c052a15b9
[2026-01-24 04:32:24 UTC]   [overlay skip] no resolvable image for crop=crop_D8_3025710764f0
[2026-01-24 04:32:24 UTC]   [overlay skip] no resolvable image for crop=crop_D8_22101f2b01d3
[2026-01-24 04:32:24 UTC]   [overlay skip] no resolvable image for crop=crop_D8_0a071dcf71bd
[2026-01-24 04:32:24 UTC]   [overlay skip] no resolvable image for crop=crop_D8_a2c2a0ac3b77
[2026-01-24 04:32:24 UTC]   [overlay skip] no resolvable image for crop=crop_D8_0bfd1412787c
[2026-01-24 04:32:24 UTC]   [overlay skip] no resolvable image for crop=crop_D8_d6b9fcfd1898


In [36]:
# -------------------------------------------------------------------
# PERSIST OUTPUTS
# -------------------------------------------------------------------
log("=" * 90)
log("PERSISTING OUTPUTS")

candidate_to_truth_match_path  = None
training_supervision_table_path = None
truth_evaluation_summary_path  = None
splits_path                    = None
matching_timing_path           = None
crop_summary_path              = None
upstream_feature_family_summary_path = None
upstream_feature_family_audit_path = None
upstream_detector_method_summary_path = None
upstream_detector_cache_audit_path = None
upstream_offtrain_summary_path = None
upstream_offtrain_method_summary_path = None
upstream_offtrain_detector_cache_audit_path = None
upstream_offtrain_feature_summary_path = None
upstream_offtrain_feature_audit_path = None
upstream_offaware_aug_summary_path = None
upstream_offaware_aug_audit_path = None

if CFG["write_candidate_to_truth_match"]:
    candidate_to_truth_match_path = safe_to_parquet(
        candidate_to_truth_match,
        TABLES_DIR / f"{RUN_ID}_candidate_to_truth_match.parquet"
    )

if CFG["write_training_supervision_table"]:
    training_supervision_table_path = safe_to_parquet(
        training_supervision_table,
        TABLES_DIR / f"{RUN_ID}_training_supervision_table.parquet"
    )

if CFG["write_truth_evaluation_summary"]:
    truth_evaluation_summary_path = safe_to_parquet(
        truth_evaluation_summary if len(truth_evaluation_summary) else pd.DataFrame(columns=[
            "annotation_id", "dataset_id", "well_id", "crop_id", "label",
            "truth_well_y_px", "truth_well_x_px", "match_status",
            "matched_union_id", "nearest_union_distance_px", "matching_registry_version",
        ]),
        TABLES_DIR / f"{RUN_ID}_truth_evaluation_summary.parquet"
    )

if CFG["write_splits"]:
    splits_path = safe_to_parquet(
        splits_df,
        TABLES_DIR / f"{RUN_ID}_splits.parquet"
    )

matching_timing_path = safe_to_parquet(
    matching_timing_df,
    TABLES_DIR / f"{RUN_ID}_matching_timing.parquet"
)
crop_summary_path = safe_to_parquet(
    crop_match_summary,
    TABLES_DIR / f"{RUN_ID}_crop_match_summary.parquet"
)

if CFG.get("write_upstream_cache_audit", True):
    if len(FEATURE_FAMILY_SUMMARY):
        upstream_feature_family_summary_path = safe_to_parquet(
            FEATURE_FAMILY_SUMMARY,
            TABLES_DIR / f"{RUN_ID}_upstream_feature_family_summary.parquet"
        )
    if len(FEATURE_FAMILY_AUDIT):
        upstream_feature_family_audit_path = safe_to_parquet(
            FEATURE_FAMILY_AUDIT,
            TABLES_DIR / f"{RUN_ID}_upstream_feature_family_audit.parquet"
        )
    if len(DETECTOR_METHOD_SUMMARY):
        upstream_detector_method_summary_path = safe_to_parquet(
            DETECTOR_METHOD_SUMMARY,
            TABLES_DIR / f"{RUN_ID}_upstream_detector_method_summary.parquet"
        )
    if len(DETECTOR_CACHE_AUDIT):
        upstream_detector_cache_audit_path = safe_to_parquet(
            DETECTOR_CACHE_AUDIT,
            TABLES_DIR / f"{RUN_ID}_upstream_detector_cache_audit.parquet"
        )

    if len(OFFTRAIN_SUMMARY):
        upstream_offtrain_summary_path = safe_to_parquet(
            OFFTRAIN_SUMMARY,
            TABLES_DIR / f"{RUN_ID}_upstream_offtrain_summary.parquet"
        )
    if len(OFFTRAIN_METHOD_SUMMARY):
        upstream_offtrain_method_summary_path = safe_to_parquet(
            OFFTRAIN_METHOD_SUMMARY,
            TABLES_DIR / f"{RUN_ID}_upstream_offtrain_method_summary.parquet"
        )
    if len(OFFTRAIN_DETECTOR_CACHE_AUDIT):
        upstream_offtrain_detector_cache_audit_path = safe_to_parquet(
            OFFTRAIN_DETECTOR_CACHE_AUDIT,
            TABLES_DIR / f"{RUN_ID}_upstream_offtrain_detector_cache_audit.parquet"
        )
    if len(OFFTRAIN_FEATURE_SUMMARY):
        upstream_offtrain_feature_summary_path = safe_to_parquet(
            OFFTRAIN_FEATURE_SUMMARY,
            TABLES_DIR / f"{RUN_ID}_upstream_offtrain_feature_summary.parquet"
        )
    if len(OFFTRAIN_FEATURE_AUDIT):
        upstream_offtrain_feature_audit_path = safe_to_parquet(
            OFFTRAIN_FEATURE_AUDIT,
            TABLES_DIR / f"{RUN_ID}_upstream_offtrain_feature_audit.parquet"
        )
    if len(OFFAUG_SUMMARY):
        upstream_offaware_aug_summary_path = safe_to_parquet(
            OFFAUG_SUMMARY,
            TABLES_DIR / f"{RUN_ID}_upstream_offaware_aug_summary.parquet"
        )
    if len(OFFAUG_AUDIT):
        upstream_offaware_aug_audit_path = safe_to_parquet(
            OFFAUG_AUDIT,
            TABLES_DIR / f"{RUN_ID}_upstream_offaware_aug_audit.parquet"
        )


manifest = {
    "run_id": RUN_ID,
    "notebook_name": "05_truth_matching_side_table_bound_fullwell_compatible_all_caches_offaware.ipynb",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "repo_root": str(REPO_ROOT),
    "matching_algorithm": CFG["matching_algorithm"],
    "matching_registry_version": CFG["matching_registry_version"],
    "truth_match_radius_px": CFG["truth_match_radius_px"],
    "negative_generation_policy": CFG["negative_generation_policy"],
    "explicit_negative_match_radius_px": CFG["explicit_negative_match_radius_px"],
    "split_registry_version": CFG["split_registry_version"],
    "sample_weight_strategy": CFG["sample_weight_strategy"],
    "counts": {
        "n_matched_positive":    n_pos,
        "n_negative":            n_neg,
        "n_false_negative":      n_fn,
        "n_hard_negative":       n_hard,
        "n_supervised_total":    int(n_total_w),
        "detection_recall":      float(n_pos / max(n_pos + n_fn, 1)),
        "positive_rate":         float(n_pos / max(n_pos + n_neg, 1)),
        "n_splits_train":        int((splits_df["split_role"] == "train").sum()),
        "n_splits_cal":          int((splits_df["split_role"] == "calibration").sum()),
        "n_splits_test":         int((splits_df["split_role"] == "test").sum()),
        "w_pos":                 float(w_pos),
        "w_neg":                 float(w_neg),
    },
    "inputs": INPUT_PATHS,
    "parity_contract": {
        "candidate_source": "NB03/04 full-well outputs",
        "crop_registry_role": "QA crop masking only",
        "recomputed_features_in_nb05": False,
        "recomputed_proposers_in_nb05": False,
        "loaded_feature_family_caches": bool(len(FEATURE_FAMILY_TABLES)),
        "loaded_detector_method_caches": bool(len(DETECTOR_METHOD_TABLES)),
        "loaded_offtrain_detector_caches": bool(len(OFFTRAIN_TABLES) or len(OFFTRAIN_METHOD_TABLES)),
        "loaded_offtrain_feature_caches": bool(len(OFFTRAIN_FEATURE_TABLES)),
        "loaded_offaware_aug_caches": bool(len(OFFAUG_TABLES)),
        "joined_offaware_aug_into_training_supervision": bool(len(globals().get("OFFAUG_JOINED_COLS", []))),
        "strict_upstream_cache_validation": bool(CFG.get("strict_upstream_cache_validation", False))
    },
    "upstream_cache_audit": UPSTREAM_CACHE_AUDIT,
    "outputs": {
        "candidate_to_truth_match":   str(candidate_to_truth_match_path) if candidate_to_truth_match_path else None,
        "training_supervision_table": str(training_supervision_table_path) if training_supervision_table_path else None,
        "truth_evaluation_summary":   str(truth_evaluation_summary_path) if truth_evaluation_summary_path else None,
        "splits":                     str(splits_path) if splits_path else None,
        "matching_timing":            str(matching_timing_path),
        "crop_match_summary":         str(crop_summary_path),
        "upstream_feature_family_summary": str(upstream_feature_family_summary_path) if upstream_feature_family_summary_path else None,
        "upstream_feature_family_audit":   str(upstream_feature_family_audit_path) if upstream_feature_family_audit_path else None,
        "upstream_detector_method_summary": str(upstream_detector_method_summary_path) if upstream_detector_method_summary_path else None,
        "upstream_detector_cache_audit":   str(upstream_detector_cache_audit_path) if upstream_detector_cache_audit_path else None,
        "upstream_offtrain_summary": str(upstream_offtrain_summary_path) if upstream_offtrain_summary_path else None,
        "upstream_offtrain_method_summary": str(upstream_offtrain_method_summary_path) if upstream_offtrain_method_summary_path else None,
        "upstream_offtrain_detector_cache_audit": str(upstream_offtrain_detector_cache_audit_path) if upstream_offtrain_detector_cache_audit_path else None,
        "upstream_offtrain_feature_summary": str(upstream_offtrain_feature_summary_path) if upstream_offtrain_feature_summary_path else None,
        "upstream_offtrain_feature_audit": str(upstream_offtrain_feature_audit_path) if upstream_offtrain_feature_audit_path else None,
        "upstream_offaware_aug_summary": str(upstream_offaware_aug_summary_path) if upstream_offaware_aug_summary_path else None,
        "upstream_offaware_aug_audit": str(upstream_offaware_aug_audit_path) if upstream_offaware_aug_audit_path else None,
        "report_dir":                 str(REPORT_DIR),
    },
}
manifest_path = write_json(manifest, MANIFEST_DIR / f"{RUN_ID}_truth_matching_manifest.json")

for k, v in manifest["outputs"].items():
    log(f"  {k}: {Path(v).name if v else None}")
log(f"  manifest: {manifest_path.name}")

[2026-01-24 04:32:25 UTC] ==========================================================================================
[2026-01-24 04:32:25 UTC] PERSISTING OUTPUTS
[2026-01-24 04:32:25 UTC]   candidate_to_truth_match: nb05_20260124T042657Z_2af2592e_candidate_to_truth_match.parquet
[2026-01-24 04:32:25 UTC]   training_supervision_table: nb05_20260124T042657Z_2af2592e_training_supervision_table.parquet
[2026-01-24 04:32:25 UTC]   truth_evaluation_summary: nb05_20260124T042657Z_2af2592e_truth_evaluation_summary.parquet
[2026-01-24 04:32:25 UTC]   splits: nb05_20260124T042657Z_2af2592e_splits.parquet
[2026-01-24 04:32:25 UTC]   matching_timing: nb05_20260124T042657Z_2af2592e_matching_timing.parquet
[2026-01-24 04:32:25 UTC]   crop_match_summary: nb05_20260124T042657Z_2af2592e_crop_match_summary.parquet
[2026-01-24 04:32:25 UTC]   upstream_feature_family_summary: nb05_20260124T042657Z_2af2592e_upstream_feature_family_summary.parquet
[2026-01-24 04:32:25 UTC]   upstream_feature_family_audit: n

In [37]:
# -------------------------------------------------------------------
# EXIT CHECKS
# -------------------------------------------------------------------
log("=" * 90)
log("EXIT CHECKS")

# ── candidate_to_truth_match (§3.7) ───────────────────────────────────────
required_match_cols = {
    "match_id", "union_id", "annotation_id", "match_status", "gt_label",
    "gt_distance_px", "matching_algorithm", "truth_match_radius_px",
    "matching_registry_version",
}
missing = sorted(required_match_cols - set(candidate_to_truth_match.columns))
assert not missing, f"candidate_to_truth_match missing: {missing}"
assert candidate_to_truth_match["match_id"].is_unique, "match_id must be unique"

_pos_check = candidate_to_truth_match[candidate_to_truth_match["match_status"] == "matched_positive"]
assert _pos_check["union_id"].is_unique,      "One-to-one violated: union_id repeated in matched_positive"
assert _pos_check["annotation_id"].is_unique, "One-to-one violated: annotation_id repeated in matched_positive"

# ── splits (§3.10) ────────────────────────────────────────────────────────
required_split_cols = {
    "split_id", "dataset_id", "well_id", "crop_id",
    "spatial_block_id", "split_role", "split_registry_version",
}
missing = sorted(required_split_cols - set(splits_df.columns))
assert not missing, f"splits missing: {missing}"
assert splits_df["crop_id"].is_unique, "Split leakage: duplicate crop_id in splits"
assert set(splits_df["split_role"]).issubset({"train", "calibration", "test"})

# ── training_supervision_table (§3.9) ─────────────────────────────────────
required_sup_cols = {
    "union_id", "split_id", "supervision_status",
    "target_binary", "target_source", "sample_weight",
}
missing = sorted(required_sup_cols - set(training_supervision_table.columns))
assert not missing, f"training_supervision_table missing: {missing}"
assert training_supervision_table["union_id"].is_unique, "training_supervision_table: duplicate union_id"

_sup_included = training_supervision_table[
    training_supervision_table["supervision_status"] == "included"
]
assert _sup_included["union_id"].is_unique, "Duplicate union_id in included supervision rows"
assert len(training_supervision_table) == len(mega_feature_table), (
    f"Row count mismatch: supervision={len(training_supervision_table)} vs mega={len(mega_feature_table)}"
)

# All supervised union_ids exist in mega_feature_table
mega_uids = set(mega_feature_table["union_id"])
sup_uids  = set(training_supervision_table["union_id"].dropna())
orphans   = sup_uids - mega_uids
assert len(orphans) == 0, f"{len(orphans)} supervised union_ids not in mega_feature_table"

log(f"\n{'=' * 90}")
log(f"NOTEBOOK 05 COMPLETE")
log(f"  candidate_to_truth_match:     {len(candidate_to_truth_match):,} rows")
log(f"    matched_positive:               {n_pos:,}")
log(f"    unmatched_candidate_negative:   {n_neg:,}")
log(f"    unmatched_truth_false_negative: {n_fn:,}")
log(f"    matched_hard_negative:          {n_hard:,}")
log(f"  detection recall (pre-classifier): {n_pos / max(n_pos + n_fn, 1):.3f}")
log(f"  positive rate in supervision:      {n_pos / max(n_pos + n_neg, 1):.4f}")
log(f"  splits: {len(splits_df)} crops → "
    f"{int((splits_df['split_role'] == 'train').sum())} train / "
    f"{int((splits_df['split_role'] == 'calibration').sum())} cal / "
    f"{int((splits_df['split_role'] == 'test').sum())} test")
log(f"  training_supervision_table:   {len(training_supervision_table):,} rows × {len(training_supervision_table.columns)} cols")
log(f"{'=' * 90}")

# ── full-well parity checks ───────────────────────────────────────────────
assert "fullwell_candidate_union" in globals(), "Expected preserved full-well candidate_union provenance"
assert "fullwell_mega_feature_table" in globals(), "Expected preserved full-well mega_feature_table provenance"
assert len(fullwell_candidate_union) >= len(candidate_union), "Masked candidate_union cannot exceed full-well candidate_union without overlap duplication"
# Feature columns beyond basic metadata must come directly from full-well mega table
_meta_like = {"union_id", "dataset_id", "well_id", "crop_id", "split_id", "split_role", "supervision_status",
              "target_binary", "target_source", "sample_weight", "annotator_status", "crop_registry_version",
              "is_completed_for_negatives", "TRUE_SPOT", "EXPLICIT_NEGATIVE"}
fullwell_feature_cols = [c for c in fullwell_mega_feature_table.columns if c not in {"dataset_id","well_id","crop_id"}]
missing_fw_cols = sorted(set(fullwell_feature_cols) - set(training_supervision_table.columns))
assert not missing_fw_cols, f"training_supervision_table missing full-well feature cols: {missing_fw_cols[:20]}"


[2026-01-24 04:32:25 UTC] ==========================================================================================
[2026-01-24 04:32:25 UTC] EXIT CHECKS
[2026-01-24 04:32:25 UTC] 
[2026-01-24 04:32:25 UTC] NOTEBOOK 05 COMPLETE
[2026-01-24 04:32:25 UTC]   candidate_to_truth_match:     867 rows
[2026-01-24 04:32:25 UTC]     matched_positive:               300
[2026-01-24 04:32:25 UTC]     unmatched_candidate_negative:   0
[2026-01-24 04:32:25 UTC]     unmatched_truth_false_negative: 3
[2026-01-24 04:32:25 UTC]     matched_hard_negative:          564
[2026-01-24 04:32:25 UTC]   detection recall (pre-classifier): 0.990
[2026-01-24 04:32:25 UTC]   positive rate in supervision:      1.0000
[2026-01-24 04:32:25 UTC]   splits: 50 crops → 37 train / 7 cal / 6 test
[2026-01-24 04:32:25 UTC]   training_supervision_table:   864 rows × 450 cols
[2026-01-24 04:32:25 UTC] ==========================================================================================
